# Tax Data Foundation: Raw to Return-Ready

**Lab 1 of 2** · 60 min core + 28 min optional · Beginner → Intermediate

Three incompatible ERP extracts, 2.4 million ledger lines, one governed GST position — and an audit trail you can reproduce on demand.

Work through the cells top to bottom. Every SQL cell is ready to run - there is nothing to copy or paste.

**Before you start**, set the notebook's role to `SYSADMIN` and pick any warehouse. Cell 1 creates `GSI_TAX_WH` and switches to it.

> Full narrative, expected results and architecture notes: **`gsi_tax_workshop_lab1_foundation.html`** — keep it open alongside this notebook.

---

> ⚠️ **Synthetic data**
>
> All data in this workshop is synthetically generated for training purposes. Entity names, ABNs, suppliers and transactions are fictitious. Tax treatments shown are simplified for illustration and are not tax advice.

---

## Step 1 — Land the raw data and look around

*~8 min* · [narrative & expected results](gsi_tax_workshop_lab1_foundation.html#step-1)

Your client's tax function spends most of its working year finding, cleaning and reconciling data before any analysis begins. We are going to skip the part where that takes six weeks. One script, one paste, and you will have three ERP extracts that describe the same business in three incompatible shapes.

### Build the environment and land the extracts

Run the whole script in one go — a single SQL cell if you are in the notebook, or a single paste into a Snowsight worksheet. It creates the warehouse, the database, seven schemas, all eight `RAW_DATA` tables, the two tax rule tables, and finishes with a verification query. Nothing else needs to be typed.

In [ ]:
-- =====================================================================
-- GSI TAX WORKSHOP — LAB 1 STEP 1 SETUP
-- Builds warehouse, database, 7 schemas and all 8 RAW_DATA tables.
-- Target: 2,400,000 GL lines. Runtime target: under 2 minutes on SMALL.
-- =====================================================================

CREATE WAREHOUSE IF NOT EXISTS GSI_TAX_WH
    WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = FALSE;

USE WAREHOUSE GSI_TAX_WH;

CREATE OR REPLACE DATABASE GSI_TAX_WORKSHOP;
USE DATABASE GSI_TAX_WORKSHOP;

CREATE SCHEMA IF NOT EXISTS RAW_DATA;
CREATE SCHEMA IF NOT EXISTS CURATED;
CREATE SCHEMA IF NOT EXISTS TAX;
CREATE SCHEMA IF NOT EXISTS AI_INSIGHTS;
CREATE SCHEMA IF NOT EXISTS AGENTS;
CREATE SCHEMA IF NOT EXISTS APPS;
CREATE SCHEMA IF NOT EXISTS AUDIT;

-- ---------------------------------------------------------------------
-- DIM_LEGAL_ENTITY — 14 rows. The crosswalk: three ERP identifiers,
-- one group entity. NULLs mean the entity does not exist in that system.
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE RAW_DATA.DIM_LEGAL_ENTITY (
    ENTITY_ID            VARCHAR(10),
    ENTITY_NAME          VARCHAR(100),
    ABN                  VARCHAR(14),
    JURISDICTION         VARCHAR(2),
    GST_REGISTERED       BOOLEAN,
    FUNCTIONAL_CURRENCY  VARCHAR(3),
    SAP_BUKRS            VARCHAR(4),
    ORACLE_LEDGER_ID     VARCHAR(10),
    LEGACY_ENTITY_CD     VARCHAR(6)
);

INSERT INTO RAW_DATA.DIM_LEGAL_ENTITY VALUES
('ENT001','Southern Cross Industrial Pty Ltd'  ,'51 824 753 556','AU',TRUE,'AUD','1010','LED_2001','AU01'),
('ENT002','Wattle Grove Logistics Pty Ltd'     ,'47 116 942 308','AU',TRUE,'AUD','1020','LED_2002','AU02'),
('ENT003','Kurrajong Manufacturing Pty Ltd'    ,'62 903 471 225','AU',TRUE,'AUD','1030','LED_2003','AU03'),
('ENT004','Boronia Retail Group Pty Ltd'       ,'19 558 260 714','AU',TRUE,'AUD','1040','LED_2004','AU04'),
('ENT005','Jarrah Point Services Pty Ltd'      ,'83 742 019 631','AU',TRUE,'AUD','1050','LED_2005','AU05'),
('ENT006','Kingsford Property Holdings Pty Ltd','35 601 887 452','AU',TRUE,'AUD','1060',NULL      ,'AU06'),
('ENT007','Tallowood Energy Pty Ltd'           ,'74 229 653 108','AU',TRUE,'AUD','1070','LED_2007',NULL ),
('ENT008','Marrickville Food Co Pty Ltd'       ,'28 470 315 967','AU',TRUE,'AUD',NULL  ,'LED_2008','AU08'),
('ENT009','Barwon Health Services Pty Ltd'     ,'91 386 724 540','AU',TRUE,'AUD','1090','LED_2009','AU09'),
('ENT010','Coolibah Finance Pty Ltd'           ,'66 052 918 379','AU',TRUE,'AUD','1100','LED_2010','AU10'),
('ENT011','Yarra Bend Education Pty Ltd'       ,'12 745 660 283','AU',TRUE,'AUD','1110','LED_2011','AU11'),
('ENT012','Nullarbor Freight Pty Ltd'          ,'58 913 407 176','AU',TRUE,'AUD',NULL  ,'LED_2012','AU12'),
('ENT013','Bass Strait Holdings Pty Ltd'       ,'40 268 591 734','AU',TRUE,'AUD','1130','LED_2013',NULL ),
('ENT014','Perisher Plant Hire Pty Ltd'        ,'77 634 128 905','AU',TRUE,'AUD','1140',NULL      ,'AU14');

-- ---------------------------------------------------------------------
-- DIM_ACCOUNT_MAPPING — 180 rows (60 per system).
-- GL uses codes 1..62 per system; mapping covers 1..60.
-- Result: ~3.2% of GL lines have no mapping (an alarming number at
-- 2.4M rows, without drowning the exception register).
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE RAW_DATA.DIM_ACCOUNT_MAPPING AS
WITH cats AS (
  SELECT ARRAY_CONSTRUCT(
    'STANDARD_GOODS','STANDARD_SERVICES','PROFESSIONAL_FEES','ENTERTAINMENT',
    'RESIDENTIAL_RENT','FINANCIAL_SUPPLY','EXPORT_SALES','FOOD_BASIC',
    'MEDICAL','EDUCATION','CAPITAL_EQUIPMENT') AS C
),
n AS (SELECT SEQ4() + 1 AS I FROM TABLE(GENERATOR(ROWCOUNT => 60))),
s AS (SELECT 'SAP' AS SYS UNION ALL SELECT 'ORACLE' UNION ALL SELECT 'LEGACY')
SELECT
    ROW_NUMBER() OVER (ORDER BY s.SYS, n.I)                       AS MAPPING_ID,
    s.SYS                                                          AS SOURCE_SYSTEM,
    CASE s.SYS
      WHEN 'SAP'    THEN '4'     || LPAD(n.I::VARCHAR, 5, '0')
      WHEN 'ORACLE' THEN '5100-' || LPAD(n.I::VARCHAR, 4, '0')
      ELSE               'AP'    || LPAD(n.I::VARCHAR, 3, '0')
    END                                                            AS NATIVE_ACCOUNT_CODE,
    'GL' || LPAD((MOD(n.I - 1, 11) * 10 + 1000)::VARCHAR, 6, '0')  AS GROUP_ACCOUNT_CODE,
    INITCAP(REPLACE(cats.C[MOD(n.I - 1, 11)]::VARCHAR, '_', ' '))  AS GROUP_ACCOUNT_NAME,
    cats.C[MOD(n.I - 1, 11)]::VARCHAR                              AS SUPPLY_CATEGORY
FROM n, s, cats;

-- ---------------------------------------------------------------------
-- DIM_SUPPLIER — 2,000 rows. 5% no ABN, 8% not GST registered.
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE RAW_DATA.DIM_SUPPLIER AS
WITH p AS (
  SELECT ARRAY_CONSTRUCT('Acacia','Redgum','Bluestone','Harbourside','Pinnacle','Meridian',
         'Clearwater','Ironbark','Sandstone','Northwind','Fairmont','Brightwater','Stirling',
         'Camperdown','Glenrowan','Yarraville','Bellevue','Crestwood','Darlington','Eastgate') AS PRE,
         ARRAY_CONSTRUCT('Supplies','Services','Group','Partners','Holdings','Industries',
         'Consulting','Logistics','Trading','Systems','Solutions','Contractors','Advisory',
         'Enterprises','Distributors') AS SUF,
         ARRAY_CONSTRUCT('AU','AU','AU','AU','AU','AU','AU','AU','AU','AU','AU','AU','AU',
         'NZ','NZ','SG','SG','US','GB','GB') AS CTRY
)
SELECT
    'SUP' || LPAD((SEQ4() + 1)::VARCHAR, 6, '0')                                  AS SUPPLIER_ID,
    p.PRE[UNIFORM(0,19,RANDOM(1001))]::VARCHAR || ' ' ||
    p.SUF[UNIFORM(0,14,RANDOM(1002))]::VARCHAR || ' Pty Ltd'                          AS SUPPLIER_NAME,
    CASE WHEN UNIFORM(1,100,RANDOM(1003)) <= 5 THEN NULL
         ELSE LPAD(UNIFORM(10,99,RANDOM(1004))::VARCHAR,2,'0') || ' ' ||
              LPAD(UNIFORM(100,999,RANDOM(1005))::VARCHAR,3,'0') || ' ' ||
              LPAD(UNIFORM(100,999,RANDOM(1006))::VARCHAR,3,'0') || ' ' ||
              LPAD(UNIFORM(100,999,RANDOM(1007))::VARCHAR,3,'0')
    END                                                                            AS SUPPLIER_ABN,
    IFF(UNIFORM(1,100,RANDOM(1008)) <=  8, FALSE, TRUE)                                AS GST_REGISTERED,
    p.CTRY[UNIFORM(0,19,RANDOM(1009))]::VARCHAR                                        AS SUPPLIER_COUNTRY
FROM TABLE(GENERATOR(ROWCOUNT => 2000)), p;

-- ---------------------------------------------------------------------
-- ERP_SAP_GL — 1,200,000 rows. Signed amounts, YYYYMMDD text dates.
-- NOTE: 1,200,000 > 1,048,576 (the Excel row limit) — deliberate.
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE RAW_DATA.ERP_SAP_GL AS
WITH p AS (
  SELECT ARRAY_CONSTRUCT('1010','1020','1030','1040','1050','1060','1070','1090','1100','1110','1130','1140') AS BK,
         ARRAY_CONSTRUCT('Freight and cartage','External consulting fee','Client hospitality',
           'Raw material purchase','Bank facility fee','Premises lease','Staff training course',
           'Software subscription','Equipment purchase','Audit and assurance fee',
           'Canteen provisions','Medical screening') AS TXT
),
b AS (
  SELECT
    'SAP' || LPAD((SEQ4() + 1)::VARCHAR, 7, '0')                                AS BELNR,
    UNIFORM(1, 8, RANDOM(1010))                                                     AS BUZEI,
    p.BK[UNIFORM(0,11,RANDOM(1011))]::VARCHAR                                       AS BUKRS,
    '4' || LPAD(UNIFORM(1, 62, RANDOM(1012))::VARCHAR, 5, '0')                      AS HKONT,
    DATEADD('day', (DATEDIFF('day','2023-06-01'::DATE, '2026-07-31'::DATE)
            * UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(1013)))::INT, '2023-06-01'::DATE)  AS D,
    UNIFORM(50, 250000, RANDOM(1014))                                               AS AMT,
    'SUP' || LPAD(UNIFORM(1, 2000, RANDOM(1015))::VARCHAR, 6, '0')                  AS LIFNR,
    CASE WHEN UNIFORM(1,100,RANDOM(1016)) <= 65 THEN 'AP_INVOICE'
         WHEN UNIFORM(1,100,RANDOM(1017)) <= 88 THEN 'AR_INVOICE'
         ELSE 'CREDIT_NOTE' END                                                 AS DOC_TYPE,
    p.TXT[UNIFORM(0,11,RANDOM(1018))]::VARCHAR                                      AS SGTXT,
    UNIFORM(1,1000,RANDOM(1019))                                                    AS R_DISTORT,
    UNIFORM(1,100,RANDOM(1020))                                                     AS R_UNREG_TAX,
    UNIFORM(1,100,RANDOM(1021))                                                     AS R_IT_WRONG
  FROM TABLE(GENERATOR(ROWCOUNT => 1200000)), p
),
j AS (
  SELECT b.*, m.SUPPLY_CATEGORY, s.GST_REGISTERED,
         -- booked tax code and rate follow the account's supply category
         CASE
           WHEN s.GST_REGISTERED = FALSE THEN IFF(b.R_UNREG_TAX <= 15, 'S1', 'S0')
           WHEN m.SUPPLY_CATEGORY IN ('RESIDENTIAL_RENT','FINANCIAL_SUPPLY')
                THEN IFF(b.R_IT_WRONG <= 10, 'S1', 'IT')
           WHEN m.SUPPLY_CATEGORY = 'EXPORT_SALES'                     THEN 'EX'
           WHEN m.SUPPLY_CATEGORY IN ('FOOD_BASIC','MEDICAL','EDUCATION') THEN 'ZF'
           ELSE 'S1'
         END                                                                    AS MWSKZ
  FROM b
  LEFT JOIN RAW_DATA.DIM_ACCOUNT_MAPPING m
         ON m.SOURCE_SYSTEM = 'SAP' AND m.NATIVE_ACCOUNT_CODE = b.HKONT
  LEFT JOIN RAW_DATA.DIM_SUPPLIER s ON s.SUPPLIER_ID = b.LIFNR
)
SELECT
    BELNR, BUZEI, BUKRS, HKONT,
    TO_VARCHAR(D, 'YYYYMMDD')                                                   AS BUDAT,
    ROUND(AMT * IFF(DOC_TYPE = 'AP_INVOICE', 1, -1), 2)                         AS DMBTR,
    ROUND(
      IFF(MWSKZ = 'S1', AMT * 0.10, 0)
      * IFF(DOC_TYPE = 'AP_INVOICE', 1, -1)
      * IFF(R_DISTORT <=  5, UNIFORM(40, 160, RANDOM(1022)) / 100.0, 1.0)
    , 2)                                                                        AS MWSTS,
    'AUD'                                                                       AS WAERS,
    MWSKZ, LIFNR, DOC_TYPE, SGTXT
FROM j;

-- ---------------------------------------------------------------------
-- ERP_ORACLE_GL — 800,000 rows. Native DATE, separate DR/CR columns.
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE RAW_DATA.ERP_ORACLE_GL AS
WITH p AS (
  SELECT ARRAY_CONSTRUCT('LED_2001','LED_2002','LED_2003','LED_2004','LED_2005','LED_2007',
           'LED_2008','LED_2009','LED_2010','LED_2011','LED_2012','LED_2013') AS LG,
         ARRAY_CONSTRUCT('Professional services rendered','Goods received','Entertainment expense',
           'Property rental','Interest and bank charges','Export freight','Training and development',
           'Capital asset acquisition','Consumables','Health services') AS TXT
),
b AS (
  SELECT
    SEQ4() + 1                                                                  AS JE_HEADER_ID,
    UNIFORM(1, 8, RANDOM(1023))                                                     AS JE_LINE_NUM,
    p.LG[UNIFORM(0,11,RANDOM(1024))]::VARCHAR                                       AS LEDGER_ID,
    '5100-' || LPAD(UNIFORM(1, 62, RANDOM(1025))::VARCHAR, 4, '0')                  AS CODE_COMBINATION,
    DATEADD('day', (DATEDIFF('day','2023-06-01'::DATE, '2026-07-31'::DATE)
            * UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(1026)))::INT, '2023-06-01'::DATE)  AS GL_DATE,
    UNIFORM(50, 250000, RANDOM(1027))                                               AS AMT,
    'SUP' || LPAD(UNIFORM(1, 2000, RANDOM(1028))::VARCHAR, 6, '0')                  AS VENDOR_ID,
    CASE WHEN UNIFORM(1,100,RANDOM(1029)) <= 65 THEN 'AP_INVOICE'
         WHEN UNIFORM(1,100,RANDOM(1030)) <= 88 THEN 'AR_INVOICE'
         ELSE 'CREDIT_NOTE' END                                                 AS DOC_TYPE,
    p.TXT[UNIFORM(0,9,RANDOM(1031))]::VARCHAR                                       AS DESCRIPTION,
    UNIFORM(1,1000,RANDOM(1032))                                                    AS R_DISTORT,
    UNIFORM(1,100,RANDOM(1033))                                                     AS R_UNREG_TAX,
    UNIFORM(1,100,RANDOM(1034))                                                     AS R_IT_WRONG
  FROM TABLE(GENERATOR(ROWCOUNT => 800000)), p
),
j AS (
  SELECT b.*, m.SUPPLY_CATEGORY, s.GST_REGISTERED,
         CASE
           WHEN s.GST_REGISTERED = FALSE THEN IFF(b.R_UNREG_TAX <= 15, 'STANDARD', 'OUT_OF_SCOPE')
           WHEN m.SUPPLY_CATEGORY IN ('RESIDENTIAL_RENT','FINANCIAL_SUPPLY')
                THEN IFF(b.R_IT_WRONG <= 10, 'STANDARD', 'EXEMPT')
           WHEN m.SUPPLY_CATEGORY = 'EXPORT_SALES'                     THEN 'ZERO_RATED'
           WHEN m.SUPPLY_CATEGORY IN ('FOOD_BASIC','MEDICAL','EDUCATION') THEN 'ZERO_RATED'
           ELSE 'STANDARD'
         END                                                                    AS TAX_CLASSIFICATION
  FROM b
  LEFT JOIN RAW_DATA.DIM_ACCOUNT_MAPPING m
         ON m.SOURCE_SYSTEM = 'ORACLE' AND m.NATIVE_ACCOUNT_CODE = b.CODE_COMBINATION
  LEFT JOIN RAW_DATA.DIM_SUPPLIER s ON s.SUPPLIER_ID = b.VENDOR_ID
)
SELECT
    JE_HEADER_ID, JE_LINE_NUM, LEDGER_ID, CODE_COMBINATION, GL_DATE,
    IFF(DOC_TYPE = 'AP_INVOICE', ROUND(AMT, 2), NULL)                           AS ENTERED_DR,
    IFF(DOC_TYPE = 'AP_INVOICE', NULL, ROUND(AMT, 2))                           AS ENTERED_CR,
    ROUND(
      IFF(TAX_CLASSIFICATION = 'STANDARD', AMT * 0.10, 0)
      * IFF(DOC_TYPE = 'AP_INVOICE', 1, -1)   -- sign must match ENTERED_DR/CR
      * IFF(R_DISTORT <= 5, UNIFORM(40, 160, RANDOM(1035)) / 100.0, 1.0)
    , 2)                                                                        AS TAX_AMOUNT,
    'AUD'                                                                       AS CURRENCY_CODE,
    TAX_CLASSIFICATION, VENDOR_ID, DOC_TYPE, DESCRIPTION
FROM j;

-- ---------------------------------------------------------------------
-- ERP_LEGACY_AP — 400,000 rows. Lower case, DD/MM/YYYY text dates,
-- GST-INCLUSIVE amounts, ~2% unparseable dates.
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE RAW_DATA.ERP_LEGACY_AP AS
WITH p AS (
  SELECT ARRAY_CONSTRUCT('AU01','AU02','AU03','AU04','AU05','AU06','AU08','AU09','AU10',
           'AU11','AU12','AU14') AS EC,
         ARRAY_CONSTRUCT('sundry purchase','contractor invoice','staff amenities',
           'office rent','finance charge','freight out','course fees','plant purchase',
           'stationery','clinic fees') AS TXT
),
b AS (
  SELECT
    'LEG' || LPAD((SEQ4() + 1)::VARCHAR, 7, '0')                                AS doc_ref,
    p.EC[UNIFORM(0,11,RANDOM(1036))]::VARCHAR                                       AS entity_cd,
    'AP' || LPAD(UNIFORM(1, 62, RANDOM(1037))::VARCHAR, 3, '0')                     AS acct,
    DATEADD('day', (DATEDIFF('day','2023-06-01'::DATE, '2026-07-31'::DATE)
            * UNIFORM(0::FLOAT, 1::FLOAT, RANDOM(1038)))::INT, '2023-06-01'::DATE)  AS D,
    UNIFORM(55, 275000, RANDOM(1039))                                               AS GROSS,
    'SUP' || LPAD(UNIFORM(1, 2000, RANDOM(1040))::VARCHAR, 6, '0')                  AS vendor_ref,
    IFF(UNIFORM(1,100,RANDOM(1041)) <= 95, 'AP_INVOICE', 'CREDIT_NOTE')             AS doc_type,
    p.TXT[UNIFORM(0,9,RANDOM(1042))]::VARCHAR                                       AS narrative,
    UNIFORM(1,1000,RANDOM(1043))                                                    AS R_DISTORT,
    UNIFORM(1,100,RANDOM(1044))                                                     AS R_UNREG_TAX,
    UNIFORM(1,100,RANDOM(1045))                                                     AS R_IT_WRONG,
    UNIFORM(1,1000,RANDOM(1046))                                                    AS R_BADDATE
  FROM TABLE(GENERATOR(ROWCOUNT => 400000)), p
),
j AS (
  SELECT b.*, m.SUPPLY_CATEGORY, s.GST_REGISTERED,
         CASE
           WHEN s.GST_REGISTERED = FALSE THEN IFF(b.R_UNREG_TAX <= 15, 'Y', 'N')
           WHEN m.SUPPLY_CATEGORY IN ('RESIDENTIAL_RENT','FINANCIAL_SUPPLY')
                THEN IFF(b.R_IT_WRONG <= 10, 'Y', 'N')
           WHEN m.SUPPLY_CATEGORY = 'EXPORT_SALES'                     THEN 'N'
           WHEN m.SUPPLY_CATEGORY IN ('FOOD_BASIC','MEDICAL','EDUCATION') THEN 'N'
           WHEN m.SUPPLY_CATEGORY IS NULL                              THEN 'X'
           ELSE 'Y'
         END                                                                    AS tax_flag
  FROM b
  LEFT JOIN RAW_DATA.DIM_ACCOUNT_MAPPING m
         ON m.SOURCE_SYSTEM = 'LEGACY' AND m.NATIVE_ACCOUNT_CODE = b.acct
  LEFT JOIN RAW_DATA.DIM_SUPPLIER s ON s.SUPPLIER_ID = b.vendor_ref
)
SELECT
    doc_ref, entity_cd, acct,
    CASE WHEN R_BADDATE <=  7 THEN '31/02/2024'
         WHEN R_BADDATE <= 14 THEN 'N/A'
         WHEN R_BADDATE <= 20 THEN ''
         ELSE TO_VARCHAR(D, 'DD/MM/YYYY') END                                   AS txn_dt,
    ROUND(GROSS, 2)                                                             AS amt_incl_tax,
    ROUND(
      IFF(tax_flag IN ('Y','X'), GROSS / 11.0, 0)
      * IFF(R_DISTORT <=  5, UNIFORM(40, 160, RANDOM(1047)) / 100.0, 1.0)
    , 2)                                                                        AS tax_amt,
    'AUD'                                                                       AS curr_cd,
    tax_flag, vendor_ref, doc_type, narrative
FROM j;

-- ---------------------------------------------------------------------
-- TAX_RULE_DETERMINATION — 16 rows. NULL = wildcard. Lower PRIORITY wins.
-- Rules 9 and 16 are the effective-dated pair: the group corrected its
-- imported-professional-services position with effect from 1 July 2024.
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE TAX.TAX_RULE_DETERMINATION (
    RULE_ID                 NUMBER,
    PRIORITY                NUMBER,
    JURISDICTION            VARCHAR(2),
    SUPPLY_CATEGORY         VARCHAR(30),
    SUPPLIER_GST_REGISTERED BOOLEAN,
    IS_CROSS_BORDER         BOOLEAN,
    TAX_TREATMENT           VARCHAR(20),
    RECOVERABLE             BOOLEAN,
    VALID_FROM              DATE,
    VALID_TO                DATE,
    RULE_DESCRIPTION        VARCHAR(200)
);

INSERT INTO TAX.TAX_RULE_DETERMINATION VALUES
( 1,10,NULL,'ENTERTAINMENT'    ,NULL ,NULL ,'NON_DEDUCTIBLE',FALSE,'2000-07-01','9999-12-31','Entertainment is taxable but the credit is denied'),
( 2,10,NULL,'RESIDENTIAL_RENT' ,NULL ,NULL ,'INPUT_TAXED'   ,FALSE,'2000-07-01','9999-12-31','Residential rent is input taxed'),
( 3,10,NULL,'FINANCIAL_SUPPLY' ,NULL ,NULL ,'INPUT_TAXED'   ,FALSE,'2000-07-01','9999-12-31','Financial supplies are input taxed'),
( 4,10,NULL,'EXPORT_SALES'     ,NULL ,NULL ,'EXPORT'        ,TRUE ,'2000-07-01','9999-12-31','Exports are GST-free'),
( 5,15,'AU','FOOD_BASIC'       ,NULL ,NULL ,'GST_FREE'      ,TRUE ,'2000-07-01','9999-12-31','Basic food is GST-free'),
( 6,15,'AU','MEDICAL'          ,NULL ,NULL ,'GST_FREE'      ,TRUE ,'2000-07-01','9999-12-31','Most medical supplies are GST-free'),
( 7,15,'AU','EDUCATION'        ,NULL ,NULL ,'GST_FREE'      ,TRUE ,'2000-07-01','9999-12-31','Most education is GST-free'),
( 8,20,NULL,'STANDARD_SERVICES',FALSE,TRUE ,'REVERSE_CHARGE',TRUE ,'2000-07-01','9999-12-31','Imported service, unregistered offshore supplier - reverse charge'),
( 9,20,NULL,'PROFESSIONAL_FEES',FALSE,TRUE ,'GST_FREE'      ,FALSE,'2000-07-01','2024-06-30','Imported professional fees - treated as out of scope (position superseded 1 Jul 2024)'),
(16,20,NULL,'PROFESSIONAL_FEES',FALSE,TRUE ,'REVERSE_CHARGE',TRUE ,'2024-07-01','9999-12-31','Imported professional fees - reverse charge applies (corrected position, from 1 Jul 2024)'),
(10,30,NULL,NULL               ,FALSE,FALSE,'GST_FREE'      ,FALSE,'2000-07-01','9999-12-31','Domestic unregistered supplier cannot charge GST - no credit'),
(11,40,NULL,'STANDARD_GOODS'   ,NULL ,NULL ,'TAXABLE'       ,TRUE ,'2000-07-01','9999-12-31','Standard goods are taxable'),
(12,40,NULL,'STANDARD_SERVICES',NULL ,NULL ,'TAXABLE'       ,TRUE ,'2000-07-01','9999-12-31','Standard services are taxable'),
(13,40,NULL,'PROFESSIONAL_FEES',NULL ,NULL ,'TAXABLE'       ,TRUE ,'2000-07-01','9999-12-31','Professional fees are taxable'),
(14,40,NULL,'CAPITAL_EQUIPMENT',NULL ,NULL ,'TAXABLE'       ,TRUE ,'2000-07-01','9999-12-31','Capital acquisitions are taxable, credit available'),
(15,99,NULL,NULL               ,NULL ,NULL ,'TAXABLE'       ,TRUE ,'2000-07-01','9999-12-31','Catch-all default - review these');

-- ---------------------------------------------------------------------
-- TAX_RULE_RATE — 19 rows. 3 jurisdictions x 6 treatments, with the
-- one standard rate split 8% -> 9% at 2024-01-01 so that every
-- (jurisdiction, treatment, date) combination is covered. The non-Australian
-- rows are unused extensibility: no worked example in either lab touches them.
-- The AU rate has not changed since 2000. The table is date-effective
-- anyway, so that if it ever does, nobody edits a query.
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE TAX.TAX_RULE_RATE (
    RATE_ID       NUMBER,
    JURISDICTION  VARCHAR(2),
    TAX_TREATMENT VARCHAR(20),
    RATE_PCT      NUMBER(5,2),
    VALID_FROM    DATE,
    VALID_TO      DATE
);

INSERT INTO TAX.TAX_RULE_RATE VALUES
( 1,'AU','TAXABLE'       ,10.00,'2000-07-01','9999-12-31'),
( 2,'AU','GST_FREE'      , 0.00,'2000-07-01','9999-12-31'),
( 3,'AU','INPUT_TAXED'   , 0.00,'2000-07-01','9999-12-31'),
( 4,'AU','EXPORT'        , 0.00,'2000-07-01','9999-12-31'),
( 5,'AU','REVERSE_CHARGE',10.00,'2000-07-01','9999-12-31'),
( 6,'AU','NON_DEDUCTIBLE',10.00,'2000-07-01','9999-12-31'),
( 7,'NZ','TAXABLE'       ,15.00,'2000-07-01','9999-12-31'),
( 8,'NZ','GST_FREE'      , 0.00,'2000-07-01','9999-12-31'),
( 9,'NZ','INPUT_TAXED'   , 0.00,'2000-07-01','9999-12-31'),
(10,'NZ','EXPORT'        , 0.00,'2000-07-01','9999-12-31'),
(11,'NZ','REVERSE_CHARGE',15.00,'2000-07-01','9999-12-31'),
(12,'NZ','NON_DEDUCTIBLE',15.00,'2000-07-01','9999-12-31'),
(13,'SG','TAXABLE'       , 8.00,'2000-07-01','2023-12-31'),
(19,'SG','TAXABLE'       , 9.00,'2024-01-01','9999-12-31'),
(14,'SG','GST_FREE'      , 0.00,'2000-07-01','9999-12-31'),
(15,'SG','INPUT_TAXED'   , 0.00,'2000-07-01','9999-12-31'),
(16,'SG','EXPORT'        , 0.00,'2000-07-01','9999-12-31'),
(17,'SG','REVERSE_CHARGE', 9.00,'2000-07-01','9999-12-31'),
(18,'SG','NON_DEDUCTIBLE', 9.00,'2000-07-01','9999-12-31');

-- ---------------------------------------------------------------------
-- Transient helper: harmonise + determine, used only to derive the
-- lodged BAS. Dropped at the end of this script.
-- ---------------------------------------------------------------------
CREATE OR REPLACE TRANSIENT TABLE RAW_DATA._TMP_DETERMINED AS
WITH u AS (
    SELECT 'SAP' AS SOURCE_SYSTEM, BUKRS AS SRC_ENTITY, HKONT AS ACCT,
           TRY_TO_DATE(BUDAT,'YYYYMMDD') AS DOC_DATE, DMBTR AS NET_AMOUNT,
           MWSTS AS BOOKED_TAX_AMOUNT, LIFNR AS SUPPLIER_ID, DOC_TYPE
    FROM RAW_DATA.ERP_SAP_GL
    UNION ALL
    SELECT 'ORACLE', LEDGER_ID, CODE_COMBINATION, GL_DATE,
           COALESCE(ENTERED_DR,0) - COALESCE(ENTERED_CR,0), TAX_AMOUNT, VENDOR_ID, DOC_TYPE
    FROM RAW_DATA.ERP_ORACLE_GL
    UNION ALL
    SELECT 'LEGACY', entity_cd, acct, TRY_TO_DATE(txn_dt,'DD/MM/YYYY'),
           amt_incl_tax - tax_amt, tax_amt, vendor_ref, doc_type
    FROM RAW_DATA.ERP_LEGACY_AP
),
h AS (
    SELECT u.*, e.ENTITY_ID, e.JURISDICTION, m.SUPPLY_CATEGORY,
           s.GST_REGISTERED AS SUPPLIER_GST_REGISTERED,
           (s.SUPPLIER_COUNTRY <> e.JURISDICTION) AS IS_CROSS_BORDER,
           DATE_TRUNC('month', u.DOC_DATE) AS TAX_PERIOD
    FROM u
    LEFT JOIN RAW_DATA.DIM_LEGAL_ENTITY e
           ON (u.SOURCE_SYSTEM='SAP'    AND e.SAP_BUKRS        = u.SRC_ENTITY)
           OR (u.SOURCE_SYSTEM='ORACLE' AND e.ORACLE_LEDGER_ID = u.SRC_ENTITY)
           OR (u.SOURCE_SYSTEM='LEGACY' AND e.LEGACY_ENTITY_CD = u.SRC_ENTITY)
    LEFT JOIN RAW_DATA.DIM_ACCOUNT_MAPPING m
           ON m.SOURCE_SYSTEM = u.SOURCE_SYSTEM AND m.NATIVE_ACCOUNT_CODE = u.ACCT
    LEFT JOIN RAW_DATA.DIM_SUPPLIER s ON s.SUPPLIER_ID = u.SUPPLIER_ID
),
d AS (
    SELECT h.ENTITY_ID, h.TAX_PERIOD, h.JURISDICTION, h.DOC_TYPE, h.NET_AMOUNT,
           r.TAX_TREATMENT, r.RECOVERABLE
    FROM h
    JOIN TAX.TAX_RULE_DETERMINATION r
      ON (r.JURISDICTION            IS NULL OR r.JURISDICTION            = h.JURISDICTION)
     AND (r.SUPPLY_CATEGORY         IS NULL OR r.SUPPLY_CATEGORY         = h.SUPPLY_CATEGORY)
     AND (r.SUPPLIER_GST_REGISTERED IS NULL OR r.SUPPLIER_GST_REGISTERED = h.SUPPLIER_GST_REGISTERED)
     AND (r.IS_CROSS_BORDER         IS NULL OR r.IS_CROSS_BORDER         = h.IS_CROSS_BORDER)
     AND h.DOC_DATE BETWEEN r.VALID_FROM AND r.VALID_TO
    WHERE h.DOC_DATE IS NOT NULL AND h.ENTITY_ID IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY h.SOURCE_SYSTEM, h.SRC_ENTITY, h.ACCT, h.DOC_DATE,
                     h.NET_AMOUNT, h.SUPPLIER_ID, h.DOC_TYPE
        ORDER BY r.PRIORITY, r.RULE_ID) = 1
)
SELECT d.ENTITY_ID, d.TAX_PERIOD, d.DOC_TYPE, d.RECOVERABLE,
       ROUND(d.NET_AMOUNT * rt.RATE_PCT / 100, 2) AS EXPECTED_TAX_AMOUNT,
       d.NET_AMOUNT
FROM d
LEFT JOIN TAX.TAX_RULE_RATE rt
       ON rt.JURISDICTION  = d.JURISDICTION
      AND rt.TAX_TREATMENT = d.TAX_TREATMENT
      AND d.TAX_PERIOD BETWEEN rt.VALID_FROM AND rt.VALID_TO;

-- ---------------------------------------------------------------------
-- FCT_BAS_LODGED — 10 AU entities x every complete month in the window.
-- ~8 entity-periods deliberately distorted (deterministic via HASH).
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE RAW_DATA.FCT_BAS_LODGED AS
WITH spine AS (
    -- One row per COMPLETE month in the GL window (2023-06-01 .. last month),
    -- so the reconciliation has no orphan periods on either side.
    SELECT TAX_PERIOD FROM (
        SELECT DATEADD('month', SEQ4(), '2023-06-01'::DATE) AS TAX_PERIOD
        FROM TABLE(GENERATOR(ROWCOUNT => 600))
    ) WHERE TAX_PERIOD < '2026-08-01'::DATE
),
au AS (SELECT ENTITY_ID FROM RAW_DATA.DIM_LEGAL_ENTITY WHERE JURISDICTION = 'AU'),
grid AS (SELECT au.ENTITY_ID, spine.TAX_PERIOD FROM au, spine),
agg AS (
    SELECT g.ENTITY_ID, g.TAX_PERIOD,
           COALESCE(ABS(SUM(CASE WHEN t.DOC_TYPE='AR_INVOICE' THEN t.NET_AMOUNT END)), 0) AS G1,
           COALESCE(ABS(SUM(CASE WHEN t.DOC_TYPE='AP_INVOICE' THEN t.NET_AMOUNT END)), 0) AS G11,
           COALESCE(ABS(SUM(CASE WHEN t.DOC_TYPE='AR_INVOICE' THEN t.EXPECTED_TAX_AMOUNT END)), 0) AS L1A,
           COALESCE(ABS(SUM(CASE WHEN t.DOC_TYPE='AP_INVOICE' AND t.RECOVERABLE
                                 THEN t.EXPECTED_TAX_AMOUNT END)), 0) AS L1B
    FROM grid g
    LEFT JOIN RAW_DATA._TMP_DETERMINED t
           ON t.ENTITY_ID = g.ENTITY_ID AND t.TAX_PERIOD = g.TAX_PERIOD
    GROUP BY g.ENTITY_ID, g.TAX_PERIOD
),
dist AS (
    SELECT agg.*,
           CASE MOD(ABS(HASH(ENTITY_ID, TAX_PERIOD)), 90)
                WHEN 0 THEN 0.85
                WHEN 1 THEN 1.12
                ELSE 1.00 END AS F
    FROM agg
)
SELECT
    ROW_NUMBER() OVER (ORDER BY ENTITY_ID, TAX_PERIOD)  AS LODGEMENT_ID,
    ENTITY_ID,
    TAX_PERIOD,
    ROUND(G1  * F, 2)                                   AS G1_TOTAL_SALES,
    ROUND(G11 * F, 2)                                   AS G11_NON_CAP_PURCHASES,
    ROUND(L1A * F, 2)                                   AS LABEL_1A_GST_ON_SALES,
    ROUND(L1B * F, 2)                                   AS LABEL_1B_GST_CREDITS,
    ROUND(L1A * F, 2) - ROUND(L1B * F, 2)               AS NET_GST,
    DATEADD('day', 21, LAST_DAY(TAX_PERIOD))            AS LODGED_DATE,
    ARRAY_CONSTRUCT('R. Whitlam','P. Nguyen','A. Kowalski','S. Fraser','M. Devi')
        [MOD(ABS(HASH(ENTITY_ID)), 5)]::VARCHAR         AS LODGED_BY
FROM dist;

DROP TABLE IF EXISTS RAW_DATA._TMP_DETERMINED;

-- ---------------------------------------------------------------------
-- DOC_TAX_INVOICE — 300 rows, selected from real SAP GL lines so the
-- join back to the curated ledger always resolves.
-- ~25% carry a narrative that disagrees with the booked category.
-- ---------------------------------------------------------------------
CREATE OR REPLACE TABLE RAW_DATA.DOC_TAX_INVOICE AS
WITH p AS (
  SELECT
    ARRAY_CONSTRUCT('STANDARD_GOODS','STANDARD_SERVICES','PROFESSIONAL_FEES','ENTERTAINMENT',
      'RESIDENTIAL_RENT','FINANCIAL_SUPPLY','EXPORT_SALES','FOOD_BASIC','MEDICAL','EDUCATION',
      'CAPITAL_EQUIPMENT') AS CATS,
    ARRAY_CONSTRUCT(
      'Steel stock purchase, mill certificate attached, delivered to Altona works.',
      'Annual software subscription billed from our United States parent entity.',
      'Monthly external audit and assurance fees for the year ended 30 June.',
      'Client dinner and drinks, six attendees, restaurant in the CBD.',
      'Residential apartment lease for a relocated employee, 12 month term.',
      'Bank facility arrangement fee on the renewed working capital line.',
      'Sea freight ex Melbourne, export consignment, FOB terms.',
      'Unprocessed fruit and vegetables supplied to the staff canteen.',
      'Corporate health screening clinic for warehouse staff.',
      'Accredited training course for the finance team, four attendees.',
      'Forklift purchase, capitalised to plant and equipment.') AS NARR,
    ARRAY_CONSTRUCT(
      'Thanks for the prompt payment, much appreciated as always.',
      'This invoice is now 45 days overdue. Please advise payment date immediately.',
      'Please note our banking details have changed from this month.',
      'We have credited the duplicate charge on your previous statement.',
      'Escalating this to our credit team - third reminder sent with no response.',
      'Happy to extend the discount if you settle within seven days.',
      'Our ABN was updated last quarter, please refresh your vendor master.',
      'Delivery was late and two units were damaged. Requesting partial credit.') AS CORR
),
base AS (
  SELECT g.BELNR, g.LIFNR, m.SUPPLY_CATEGORY, s.SUPPLIER_ABN,
         UNIFORM(1,100,RANDOM(1048)) AS MISCODE_R,
         UNIFORM(1,10,RANDOM(1049))  AS SHIFT_R,
         UNIFORM(1,100,RANDOM(1050)) AS ABN_R,
         UNIFORM(0,7,RANDOM(1051))   AS CORR_R,
         ROW_NUMBER() OVER (ORDER BY g.BELNR) AS RN
  FROM RAW_DATA.ERP_SAP_GL g
  JOIN RAW_DATA.DIM_ACCOUNT_MAPPING m
    ON m.SOURCE_SYSTEM='SAP' AND m.NATIVE_ACCOUNT_CODE = g.HKONT
  JOIN RAW_DATA.DIM_SUPPLIER s ON s.SUPPLIER_ID = g.LIFNR
  WHERE g.DOC_TYPE = 'AP_INVOICE'
  LIMIT 300
),
idx AS (
  SELECT base.*,
         ARRAY_POSITION(base.SUPPLY_CATEGORY::VARIANT, p.CATS) AS CAT_IDX, p.NARR, p.CORR
  FROM base, p
)
SELECT
    'DOC' || LPAD(RN::VARCHAR, 6, '0')                                       AS DOC_ID,
    'SAP'                                                                    AS SOURCE_SYSTEM,
    BELNR                                                                    AS SOURCE_DOC_REF,
    LIFNR                                                                    AS SUPPLIER_ID,
    SUPPLY_CATEGORY                                                          AS BOOKED_SUPPLY_CATEGORY,
    NARR[IFF(MISCODE_R <= 75, CAT_IDX, MOD(CAT_IDX + SHIFT_R, 11))]::VARCHAR AS INVOICE_NARRATIVE,
    CASE WHEN ABN_R <= 70 AND SUPPLIER_ABN IS NOT NULL
         THEN 'Tax invoice. Supplier ABN ' || SUPPLIER_ABN ||
              '. GST included in the total shown.'
         ELSE 'Invoice total as shown. Please remit within 30 days.'
    END                                                                      AS SUPPLIER_REMARKS,
    CORR[CORR_R]::VARCHAR                                                    AS CORRESPONDENCE_TEXT
FROM idx;

-- ---------------------------------------------------------------------
-- VERIFICATION — object names and row counts.
-- Expected: 1,200,000 / 800,000 / 400,000 / 14 / 180 / 2,000 / 532 /
--           300 / 16 / 19
-- ---------------------------------------------------------------------
SELECT 'RAW_DATA.ERP_SAP_GL'          AS OBJECT_NAME, COUNT(*) AS ROW_COUNT FROM RAW_DATA.ERP_SAP_GL
UNION ALL SELECT 'RAW_DATA.ERP_ORACLE_GL',      COUNT(*) FROM RAW_DATA.ERP_ORACLE_GL
UNION ALL SELECT 'RAW_DATA.ERP_LEGACY_AP',      COUNT(*) FROM RAW_DATA.ERP_LEGACY_AP
UNION ALL SELECT 'RAW_DATA.DIM_LEGAL_ENTITY',   COUNT(*) FROM RAW_DATA.DIM_LEGAL_ENTITY
UNION ALL SELECT 'RAW_DATA.DIM_ACCOUNT_MAPPING',COUNT(*) FROM RAW_DATA.DIM_ACCOUNT_MAPPING
UNION ALL SELECT 'RAW_DATA.DIM_SUPPLIER',       COUNT(*) FROM RAW_DATA.DIM_SUPPLIER
UNION ALL SELECT 'RAW_DATA.FCT_BAS_LODGED',     COUNT(*) FROM RAW_DATA.FCT_BAS_LODGED
UNION ALL SELECT 'RAW_DATA.DOC_TAX_INVOICE',    COUNT(*) FROM RAW_DATA.DOC_TAX_INVOICE
UNION ALL SELECT 'TAX.TAX_RULE_DETERMINATION',  COUNT(*) FROM TAX.TAX_RULE_DETERMINATION
UNION ALL SELECT 'TAX.TAX_RULE_RATE',           COUNT(*) FROM TAX.TAX_RULE_RATE
ORDER BY OBJECT_NAME;

> ✅ **What just happened**
>
> You provisioned compute, created a database and generated 2,400,000 ledger lines in about a minute. No server was sized, no storage was allocated, no index was built and there is nothing to maintain.

Check the verification output against these expected counts before continuing.

| Object | Expected rows |
|---|---|
| `RAW_DATA.ERP_SAP_GL` | 1,200,000 |
| `RAW_DATA.ERP_ORACLE_GL` | 800,000 |
| `RAW_DATA.ERP_LEGACY_AP` | 400,000 |
| `RAW_DATA.DIM_LEGAL_ENTITY` | 14 |
| `RAW_DATA.DIM_ACCOUNT_MAPPING` | 180 |
| `RAW_DATA.DIM_SUPPLIER` | 2,000 |
| `RAW_DATA.FCT_BAS_LODGED` | 532 |
| `RAW_DATA.DOC_TAX_INVOICE` | 300 |
| `TAX.TAX_RULE_DETERMINATION` | 16 |
| `TAX.TAX_RULE_RATE` | 19 |

### Look at the three shapes side by side

One query, nine rows, three source systems. Everything is cast to `VARCHAR` so that nothing is silently coerced on the way to your screen — you see exactly what landed.

In [ ]:
-- Three source systems, nine rows, everything cast to VARCHAR so nothing
-- is silently coerced. Read down the DATE_AS_LANDED and AMOUNT columns.
SELECT * FROM (
    SELECT 'SAP'    AS SOURCE_SYSTEM,
           BELNR::VARCHAR              AS DOC_REF,
           BUKRS::VARCHAR              AS ENTITY_AS_LANDED,
           BUDAT::VARCHAR              AS DATE_AS_LANDED,
           DMBTR::VARCHAR              AS AMOUNT_1,
           NULL::VARCHAR               AS AMOUNT_2,
           MWSTS::VARCHAR              AS TAX_AS_BOOKED,
           MWSKZ::VARCHAR              AS TAX_CODE
    FROM RAW_DATA.ERP_SAP_GL LIMIT 3)
UNION ALL
SELECT * FROM (
    SELECT 'ORACLE',
           JE_HEADER_ID::VARCHAR,
           LEDGER_ID::VARCHAR,
           GL_DATE::VARCHAR,
           ENTERED_DR::VARCHAR,
           ENTERED_CR::VARCHAR,
           TAX_AMOUNT::VARCHAR,
           TAX_CLASSIFICATION::VARCHAR
    FROM RAW_DATA.ERP_ORACLE_GL LIMIT 3)
UNION ALL
SELECT * FROM (
    SELECT 'LEGACY',
           doc_ref::VARCHAR,
           entity_cd::VARCHAR,
           txn_dt::VARCHAR,
           amt_incl_tax::VARCHAR,
           NULL::VARCHAR,
           tax_amt::VARCHAR,
           tax_flag::VARCHAR
    FROM RAW_DATA.ERP_LEGACY_AP LIMIT 3)
ORDER BY SOURCE_SYSTEM, DOC_REF;

Read down two columns. `DATE_AS_LANDED` returns `20250513`, `2023-11-02` and `17/02/2026`. The amount columns are worse: SAP posts a single signed figure, Oracle splits debits and credits into two columns and leaves the other null, and the legacy system posts one unsigned figure that already includes the GST.

### The same business concept, three names

| Business concept | ERP_SAP_GL | ERP_ORACLE_GL | ERP_LEGACY_AP |
|---|---|---|---|
| Document reference | `BELNR` | `JE_HEADER_ID` | `doc_ref` |
| Entity identifier | `BUKRS` — `1010` | `LEDGER_ID` — `LED_2001` | `entity_cd` — `AU01` |
| Posting date | `BUDAT` — text, `YYYYMMDD` | `GL_DATE` — native `DATE` | `txn_dt` — text, `DD/MM/YYYY` |
| Account code | `HKONT` — `400001` | `CODE_COMBINATION` — `5100-0001` | `acct` — `AP001` |
| Amount | `DMBTR` — signed, GST-exclusive | `ENTERED_DR` / `ENTERED_CR` — two columns, unsigned | `amt_incl_tax` — GST-inclusive |
| Tax amount | `MWSTS` | `TAX_AMOUNT` | `tax_amt` |
| Tax code | `MWSKZ` — `S1`, `S0`, `IT`, `EX`, `ZF` | `TAX_CLASSIFICATION` — `STANDARD`, `EXEMPT`, `ZERO_RATED`, `OUT_OF_SCOPE` | `tax_flag` — `Y`, `N`, `X` |
| Supplier | `LIFNR` | `VENDOR_ID` | `vendor_ref` |
| Narrative | `SGTXT` | `DESCRIPTION` | `narrative` |
| Currency | `WAERS` | `CURRENCY_CODE` | `curr_cd` |

> 💡 **Nothing here has been cleaned**
>
> That is deliberate. This is what the data actually looks like when it lands from three systems that were bought, built and acquired at different times by different people.

- Compute, a database and 2.4 million ledger lines from a single script
- Three date formats, three amount conventions and three tax code vocabularies for one group
- The raw layer holds data exactly as landed — no conformance, no interpretation

---

## Step 2 — Prove you cannot answer the question

*~8 min* · [narrative & expected results](gsi_tax_workshop_lab1_foundation.html#step-2)

A partner puts a question to you that any client would ask: **what was the group's GST payable for the last full quarter, by entity?** It is a completely reasonable question. Right now it is unanswerable. The job of this step is to prove that in three queries, not to catalogue every flaw in the data.

### The plausible wrong answer

SAP is the biggest of the three systems and the one the group thinks of as the general ledger. Answer the question against SAP alone.

In [ ]:
-- The partner's question, answered against SAP alone.
-- GST booked by company code for the last three complete months.
SELECT
    g.BUKRS                    AS SAP_COMPANY_CODE,
    e.ENTITY_NAME,
    COUNT(*)                   AS GL_LINES_SCANNED,
    ROUND(SUM(g.MWSTS), 2)     AS GST_BOOKED_AUD
FROM RAW_DATA.ERP_SAP_GL g
LEFT JOIN RAW_DATA.DIM_LEGAL_ENTITY e
       ON e.SAP_BUKRS = g.BUKRS
WHERE TRY_TO_DATE(g.BUDAT, 'YYYYMMDD') >= DATEADD('month', -3, DATE_TRUNC('month', CURRENT_DATE()))
  AND TRY_TO_DATE(g.BUDAT, 'YYYYMMDD') <  DATE_TRUNC('month', CURRENT_DATE())
GROUP BY 1, 2
ORDER BY GST_BOOKED_AUD DESC;

That returns in about a second: a clean, confident, formatted number per company code, with **96,025 ledger lines scanned** and **$176,904,538** of GST across **12 rows**.

> ✅ **The scale moment**
>
> `ERP_SAP_GL` holds 1,200,000 rows. Microsoft Excel stops at 1,048,576 — one of your three source systems, on its own, will not open in the tool your team uses today.

> ⚠️ **And it is still the wrong answer**
>
> The group has 14 legal entities. That result has 12 rows, because two entities do not exist in SAP at all — Marrickville Food Co and Nullarbor Freight are booked in Oracle and in the legacy system.

> 💡 **Free platform moment — no query needed**
>
> With those results still on screen, click a column header in the Snowsight results pane. Snowsight shows the value distribution, the null count and the distinct count for that column immediately.

### So stack the three systems together

The obvious next move is to union the three extracts on the columns that look equivalent. Try it.

> ⚠️ **This cell is meant to fail**
>
> Run it anyway. The error is the teaching point, and the next cell fixes it.

In [ ]:
-- The obvious next move: stack the three extracts on the columns that
-- look equivalent. Oracle splits the amount across two columns, so the
-- tax code slides into the amount slot. This query FAILS. That is the point.
SELECT BUDAT  AS DOC_DATE, DMBTR        AS AMOUNT, MWSKZ  AS TAX_CODE
FROM RAW_DATA.ERP_SAP_GL
UNION ALL
SELECT GL_DATE, ENTERED_DR, ENTERED_CR
FROM RAW_DATA.ERP_ORACLE_GL
UNION ALL
SELECT txn_dt, amt_incl_tax, tax_flag
FROM RAW_DATA.ERP_LEGACY_AP;

-- Numeric value 'S0' is not recognized

It fails. `Numeric value 'S0' is not recognized`. There is no way to line these three extracts up by position, and there are four separate reasons why.

- **Date types.** One column is a real `DATE`; the other two are text in two different conventions. `20250513` and `13/05/2025` cannot be compared, sorted or filtered against each other until both are parsed, and roughly 2% of the legacy dates are not valid dates at all.
- **Amount conventions.** SAP posts one signed GST-exclusive figure. Oracle posts unsigned amounts split across a debit column and a credit column, one of which is always null. The legacy system posts a single unsigned GST-inclusive figure, so the taxable value has to be derived by removing one eleventh before anything can be added up.
- **Entity keys.** `1010`, `LED_2001` and `AU01` are the same company. Nothing in the three extracts connects them; only `DIM_LEGAL_ENTITY` does, and it is also the only place that records which entities are absent from which system.
- **Account codes.** `400001`, `5100-0001` and `AP001` describe the same expense. Without the mapping there is no supply category, and without a supply category no tax treatment can be determined.

### One query, the whole diagnosis

Rather than run a separate count for each problem, ask for all of them at once: one row per defect, with a count and a percentage of the relevant population.

In [ ]:
-- One query, one row per defect, the whole diagnosis. Scans all 2,400,000
-- ledger lines plus the supplier and entity masters.
WITH gl AS (
    SELECT 'SAP' AS SOURCE_SYSTEM, HKONT AS NATIVE_ACCOUNT_CODE,
           LIFNR AS SUPPLIER_ID, MWSTS AS BOOKED_TAX
    FROM RAW_DATA.ERP_SAP_GL
    UNION ALL
    SELECT 'ORACLE', CODE_COMBINATION, VENDOR_ID, TAX_AMOUNT
    FROM RAW_DATA.ERP_ORACLE_GL
    UNION ALL
    SELECT 'LEGACY', acct, vendor_ref, tax_amt
    FROM RAW_DATA.ERP_LEGACY_AP
)
SELECT 'Legacy dates that will not parse'                    AS ISSUE,
       COUNT(*)                                              AS AFFECTED,
       ROUND(100.0 * COUNT(*)
             / (SELECT COUNT(*) FROM RAW_DATA.ERP_LEGACY_AP), 2) AS PCT_OF_POPULATION,
       'ERP_LEGACY_AP lines'                                 AS POPULATION
FROM RAW_DATA.ERP_LEGACY_AP
WHERE TRY_TO_DATE(txn_dt, 'DD/MM/YYYY') IS NULL

UNION ALL
SELECT 'GL lines with no account mapping', COUNT(*),
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM gl), 2), 'All GL lines'
FROM gl g
LEFT JOIN RAW_DATA.DIM_ACCOUNT_MAPPING m
       ON m.SOURCE_SYSTEM = g.SOURCE_SYSTEM
      AND m.NATIVE_ACCOUNT_CODE = g.NATIVE_ACCOUNT_CODE
WHERE m.MAPPING_ID IS NULL

UNION ALL
SELECT 'GL lines with GST booked by an unregistered supplier', COUNT(*),
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM gl), 2), 'All GL lines'
FROM gl g
JOIN RAW_DATA.DIM_SUPPLIER s ON s.SUPPLIER_ID = g.SUPPLIER_ID
WHERE s.GST_REGISTERED = FALSE AND ABS(g.BOOKED_TAX) > 0

UNION ALL
SELECT 'Suppliers with no ABN on file', COUNT(*),
       ROUND(100.0 * COUNT(*)
             / (SELECT COUNT(*) FROM RAW_DATA.DIM_SUPPLIER), 2), 'Supplier master records'
FROM RAW_DATA.DIM_SUPPLIER
WHERE SUPPLIER_ABN IS NULL

UNION ALL
SELECT 'Entities missing from at least one ERP', COUNT(*),
       ROUND(100.0 * COUNT(*)
             / (SELECT COUNT(*) FROM RAW_DATA.DIM_LEGAL_ENTITY), 2), 'Group legal entities'
FROM RAW_DATA.DIM_LEGAL_ENTITY
WHERE SAP_BUKRS IS NULL OR ORACLE_LEDGER_ID IS NULL OR LEGACY_ENTITY_CD IS NULL

ORDER BY AFFECTED DESC;

Expect results close to these. The exact figures move slightly because the transaction window extends to the current date.

| Issue | Affected | Percentage | Population |
|---|---|---|---|
| GL lines with no account mapping | 77,484 | 3.23% | All GL lines |
| GL lines with GST booked by an unregistered supplier | 31,039 | 1.29% | All GL lines |
| Legacy dates that will not parse | 8,088 | 2.02% | `ERP_LEGACY_AP` lines |
| Suppliers with no ABN on file | 102 | 5.10% | Supplier master records |
| Entities missing from at least one ERP | 6 | 42.86% | Group legal entities |

Read the first row out loud. **77,484 ledger lines carry an account code that nobody has mapped.** Those lines have no supply category, so no rule can reach them, so their GST treatment is unknown. That is not a data hygiene footnote; it is an unquantified position in a lodged return.

### The full defect inventory

The scorecard covers the defects that block the question outright. These are all of them, including the ones that only become visible once the ledger is conformed and the rules are applied in the next three steps.

| # | Defect | Volume | Where it is dealt with |
|---|---|---|---|
| 1 | Three date formats: `YYYYMMDD` text, native `DATE`, `DD/MM/YYYY` text | all 2,400,000 | Step 3 date normalisation |
| 2 | Malformed legacy dates | 8,088 | `IS_DATE_PARSED`, exception E2, `TRY_TO_DATE` |
| 3 | Three amount conventions: signed / debit-credit pair / GST-inclusive | all 2,400,000 | Step 3 sign and gross-up conformance |
| 4 | Three entity identifier schemes | all 2,400,000 | Step 3 entity resolution |
| 5 | Account codes absent from the mapping | 77,484 (3.23%) | `IS_ACCOUNT_MAPPED`, exception E1 |
| 6 | Suppliers with no ABN | 102 suppliers | Exception E3, input tax credit exposure |
| 7 | Unregistered suppliers charging GST | 31,039 lines | Exception E4 |
| 8 | Booked GST differs from expected GST by more than $1 | 107,808 lines | Exception E5 |
| 9 | Credits claimed on non-recoverable treatments | 36,718 lines | Exception E6 |
| 10 | Cross-border services with no reverse charge | 11,754 lines | Exception E7 |
| 11 | Lodged BAS diverges from the computed position | 18 entity-periods | Step 5 reconciliation and audit |
| 12 | Entities missing from one or two source systems | 6 of 14 | Step 3 outer-join reasoning |
| 13 | Booked supply category disagrees with the invoice narrative | ~25% of 300 documents | Lab 2 Step 2 miscoding detection |

> 💡 **The work is not the analysis**
>
> The partner's question needed one `GROUP BY`. Everything that stands between you and a defensible answer to it is conformance: parsing dates, resolving entities, mapping accounts, normalising signs.

---

## Step 3 — Conform three ERPs into one governed ledger

*~14 min* · [narrative & expected results](gsi_tax_workshop_lab1_foundation.html#step-3)

In Step 2 you proved that no single source system can answer the question. This step fixes that permanently. One object, every line from every system in one shape, with the dimensions attached and the data-quality problems labelled rather than hidden.

- Parse three different date formats without the query dying on the bad rows.
- Conform three amount conventions into one signed net amount.
- Resolve three entity identifier schemes to one group entity.
- Build one Dynamic Table that keeps itself up to date as the source systems change.

### Block 1 — the date problem, and the one-word fix

The legacy AP system stores its document date as text in `DD/MM/YYYY`. About 2% of those 400,000 values cannot be parsed at all: some are `'N/A'`, some are empty strings, and some are dates that look plausible but do not exist — `'31/02/2024'` among them. Run both statements below in order.

> ⚠️ **This cell is meant to fail**
>
> Run it anyway. The error is the teaching point, and the next cell fixes it.

In [ ]:
-- 1. The strict parser. This is EXPECTED TO FAIL - that is the lesson.
--    TO_DATE does not skip the bad rows and it does not return NULL for
--    them. One unparseable value out of 400,000 fails the entire query.
SELECT MIN(TO_DATE(txn_dt, 'DD/MM/YYYY')) AS EARLIEST_DOC_DATE
FROM RAW_DATA.ERP_LEGACY_AP;

In [ ]:
-- 2. The tolerant parser. Same column, same format mask, one word
--    different. Every row survives. The values that cannot be parsed
--    arrive as NULL, which means they can be counted, reported and
--    chased instead of taking the pipeline down with them.
SELECT txn_dt                            AS RAW_VALUE,
       TRY_TO_DATE(txn_dt, 'DD/MM/YYYY') AS TOLERANT_PARSE,
       COUNT(*)                          AS LINE_COUNT
FROM RAW_DATA.ERP_LEGACY_AP
WHERE TRY_TO_DATE(txn_dt, 'DD/MM/YYYY') IS NULL
GROUP BY 1, 2
ORDER BY 3 DESC;

Statement 1 stops with `Can't parse '' as date with format 'DD/MM/YYYY'`. One empty string, out of 400,000 rows, and there is no result at all — no partial answer, no list of the rows at fault. Statement 2 returns the three shapes that break it:

| Raw value | Tolerant parse | Lines | Why it fails |
|---|---|---|---|
| `N/A` | `NULL` | 2,799 | Not a date at all — an operator typed a placeholder. |
| `31/02/2024` | `NULL` | 2,871 | Correctly formatted and still impossible: February has no 31st. |
| *(empty string)* | `NULL` | 2,418 | The field was never populated. |
| **Total unparseable** |  | **8,088** | 2.02% of the legacy ledger. |

The middle case is the instructive one. `31/02/2024` passes any format check, any regular expression and any length validation you might write — it is only invalid because that day does not exist. A calendar-aware parser catches it; pattern matching never will.

> 💡 **The distinction that matters**
>
> `TO_DATE` asserts that the data is clean. `TRY_TO_DATE` asserts nothing and tells you where it is not.

### What has to be conformed, and why

The three systems disagree on more than dates. They disagree about how an amount is signed, what an amount includes, and how a legal entity is named. None of that can be left to the person writing the next query.

| Source | Document date | Amount convention | Entity identifier |
|---|---|---|---|
| **SAP** | `BUDAT` — text, `YYYYMMDD` | Signed amount in `DMBTR`, tax carried separately in `MWSTS` | `BUKRS` company code |
| **Oracle** | `GL_DATE` — native `DATE`, no parsing needed | Separate debit and credit columns, netted as `ENTERED_DR - ENTERED_CR` | `LEDGER_ID` |
| **Legacy AP** | `txn_dt` — text, `DD/MM/YYYY`, ~2% unparseable | GST-**inclusive** gross; net recovered as `amt_incl_tax - tax_amt` | `entity_cd` |

The amount conventions are the quietest of the three defects and the most dangerous. A signed amount, a debit-minus-credit pair and a GST-inclusive gross are all "the amount" to the system that produced them. Add them together untouched and the total is meaningless — and it will still look like a number, which is why nobody catches it. Each branch of the union below normalises to the same definition: `NET_AMOUNT` is always net of GST, and `BOOKED_TAX_AMOUNT` is always the tax the source system actually posted.

The entity crosswalk works the same way. `DIM_LEGAL_ENTITY` holds one row per group entity carrying all three identifiers — `SAP_BUKRS`, `ORACLE_LEDGER_ID` and `LEGACY_ENTITY_CD`. The `ent_xwalk` CTE unpivots that once into `(SOURCE_SYSTEM, SRC_ENTITY) → ENTITY_ID`, so the union can be joined a single time on a single pair of columns instead of carrying a three-branch `OR` condition. One group entity, three names, resolved in one place.

### Block 2 — build the governed ledger

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE CURATED.GL_HARMONISED
    TARGET_LAG = '1 hour'
    WAREHOUSE  = GSI_TAX_WH
    REFRESH_MODE = AUTO
AS
WITH ent_xwalk AS (
    -- Unpivot the crosswalk once: three ERP identifiers, one group entity.
    SELECT ENTITY_ID, 'SAP'    AS SOURCE_SYSTEM, SAP_BUKRS        AS SRC_ENTITY
      FROM RAW_DATA.DIM_LEGAL_ENTITY WHERE SAP_BUKRS        IS NOT NULL
    UNION ALL
    SELECT ENTITY_ID, 'ORACLE', ORACLE_LEDGER_ID FROM RAW_DATA.DIM_LEGAL_ENTITY
     WHERE ORACLE_LEDGER_ID IS NOT NULL
    UNION ALL
    SELECT ENTITY_ID, 'LEGACY', LEGACY_ENTITY_CD FROM RAW_DATA.DIM_LEGAL_ENTITY
     WHERE LEGACY_ENTITY_CD IS NOT NULL
),
u AS (
    -- SAP: signed amounts, YYYYMMDD text dates
    SELECT 'SAP' AS SOURCE_SYSTEM, BELNR AS SOURCE_DOC_REF, BUZEI AS SOURCE_LINE_NUM,
           BUKRS AS SRC_ENTITY, HKONT AS NATIVE_ACCOUNT_CODE,
           TRY_TO_DATE(BUDAT, 'YYYYMMDD') AS DOC_DATE, DOC_TYPE,
           DMBTR AS NET_AMOUNT, MWSTS AS BOOKED_TAX_AMOUNT,
           WAERS AS CURRENCY, MWSKZ AS BOOKED_TAX_CODE,
           LIFNR AS SUPPLIER_ID, SGTXT AS LINE_NARRATIVE
    FROM RAW_DATA.ERP_SAP_GL
    UNION ALL
    -- Oracle: native DATE, separate DR/CR columns
    SELECT 'ORACLE', JE_HEADER_ID::VARCHAR, JE_LINE_NUM,
           LEDGER_ID, CODE_COMBINATION,
           GL_DATE, DOC_TYPE,
           COALESCE(ENTERED_DR, 0) - COALESCE(ENTERED_CR, 0), TAX_AMOUNT,
           CURRENCY_CODE, TAX_CLASSIFICATION,
           VENDOR_ID, DESCRIPTION
    FROM RAW_DATA.ERP_ORACLE_GL
    UNION ALL
    -- Legacy: DD/MM/YYYY text dates, GST-INCLUSIVE amounts (gross up removed)
    SELECT 'LEGACY', doc_ref, 1,
           entity_cd, acct,
           TRY_TO_DATE(txn_dt, 'DD/MM/YYYY'), doc_type,
           amt_incl_tax - tax_amt, tax_amt,
           curr_cd, tax_flag,
           vendor_ref, narrative
    FROM RAW_DATA.ERP_LEGACY_AP
)
SELECT
    u.SOURCE_SYSTEM || '-' || u.SOURCE_DOC_REF || '-' || u.SOURCE_LINE_NUM::VARCHAR AS GL_LINE_ID,
    u.SOURCE_SYSTEM, u.SOURCE_DOC_REF, u.SOURCE_LINE_NUM,
    x.ENTITY_ID, e.ENTITY_NAME, e.JURISDICTION, e.ABN AS ENTITY_ABN,
    u.DOC_DATE,
    DATE_TRUNC('month', u.DOC_DATE)                              AS TAX_PERIOD,
    u.DOC_TYPE,
    u.NATIVE_ACCOUNT_CODE,
    m.GROUP_ACCOUNT_CODE, m.GROUP_ACCOUNT_NAME, m.SUPPLY_CATEGORY,
    u.SUPPLIER_ID, s.SUPPLIER_NAME, s.SUPPLIER_ABN,
    s.GST_REGISTERED                                             AS SUPPLIER_GST_REGISTERED,
    s.SUPPLIER_COUNTRY,
    (s.SUPPLIER_COUNTRY <> e.JURISDICTION)                       AS IS_CROSS_BORDER,
    u.CURRENCY,
    u.NET_AMOUNT,
    u.BOOKED_TAX_AMOUNT,
    u.NET_AMOUNT + u.BOOKED_TAX_AMOUNT                           AS GROSS_AMOUNT,
    u.BOOKED_TAX_CODE,
    u.LINE_NARRATIVE,
    -- conformance flags: is the DATA complete? (not: is the TAX right?)
    (u.DOC_DATE          IS NOT NULL)                            AS IS_DATE_PARSED,
    (x.ENTITY_ID         IS NOT NULL)                            AS IS_ENTITY_RESOLVED,
    (m.SUPPLY_CATEGORY   IS NOT NULL)                            AS IS_ACCOUNT_MAPPED,
    (s.SUPPLIER_ID       IS NOT NULL)                            AS IS_SUPPLIER_RESOLVED
FROM u
LEFT JOIN ent_xwalk x
       ON x.SOURCE_SYSTEM = u.SOURCE_SYSTEM AND x.SRC_ENTITY = u.SRC_ENTITY
LEFT JOIN RAW_DATA.DIM_LEGAL_ENTITY e   ON e.ENTITY_ID = x.ENTITY_ID
LEFT JOIN RAW_DATA.DIM_ACCOUNT_MAPPING m
       ON m.SOURCE_SYSTEM = u.SOURCE_SYSTEM AND m.NATIVE_ACCOUNT_CODE = u.NATIVE_ACCOUNT_CODE
LEFT JOIN RAW_DATA.DIM_SUPPLIER s       ON s.SUPPLIER_ID = u.SUPPLIER_ID;

Notice the shape: three `SELECT` branches unioned inside a CTE, then the dimension joins applied **once** over the union rather than repeated per branch. Add a fourth source system and you add one branch and one crosswalk row. Nothing else in the statement changes.

Notice also that every dimension join is a `LEFT JOIN`. An inner join here would quietly delete the lines with an unmapped account code, and those are precisely the lines the tax team needs to see. The four conformance flags at the bottom exist so that incompleteness is recorded as a fact about the row rather than resolved by dropping it.

> 💡 **The account mapping is a JOIN, not a CASE statement**
>
> `SUPPLY_CATEGORY` arrives from `DIM_ACCOUNT_MAPPING` through a join on `(SOURCE_SYSTEM, NATIVE_ACCOUNT_CODE)`. It would have been shorter to write a `CASE` expression listing account codes inline, and it would have been wrong.

One more thing this table deliberately does not contain: any tax logic. There are no rates, no treatments and no judgements about whether GST was handled correctly. That is because the same conformed ledger feeds statutory reporting, management reporting and tax, and the moment a tax assumption is baked into it, the other two consumers inherit it silently.

> ✅ **What one CREATE statement just bought you**
>
> That single statement produced a working pipeline: it unions three source systems, resolves four dimensions and labels four data-quality conditions across **2,400,000 rows** — and it keeps itself up to date as the sources change. There is no scheduler to configure, no orchestration tool to install, no dependency graph to maintain by hand and nothing to deploy.

Three things worth doing on screen before you move on, none of which needs a query. First, open **Monitoring → Dynamic Tables** in Snowsight: the refresh history and the dependency graph are already there, because the pipeline draws its own diagram. Second, when you run Block 3 below, open the **Query Profile** tab on the result — a visual execution plan with partition pruning statistics, for free, on a query you were running anyway. Third, run Block 3 a second time: it returns instantly from the result cache and consumes no compute at all.

### Block 3 — one query that verifies the whole build

In [ ]:
SELECT COUNT(*)                                                    AS TOTAL_ROWS,
       COUNT(DISTINCT GL_LINE_ID)                                  AS DISTINCT_LINES,
       ROUND(100.0 * SUM(IFF(IS_DATE_PARSED, 1, 0))      / COUNT(*), 2) AS PCT_DATE_OK,
       ROUND(100.0 * SUM(IFF(IS_ENTITY_RESOLVED, 1, 0))  / COUNT(*), 2) AS PCT_ENTITY_OK,
       ROUND(100.0 * SUM(IFF(IS_ACCOUNT_MAPPED, 1, 0))   / COUNT(*), 2) AS PCT_ACCOUNT_OK,
       ROUND(100.0 * SUM(IFF(IS_SUPPLIER_RESOLVED, 1, 0))/ COUNT(*), 2) AS PCT_SUPPLIER_OK
FROM CURATED.GL_HARMONISED;

| Measure | Result | What it tells you |
|---|---|---|
| `TOTAL_ROWS` | **2,400,000** | 1,200,000 SAP + 800,000 Oracle + 400,000 legacy. Nothing lost. |
| `DISTINCT_LINES` | **2,400,000** | Identical to the row count, so no dimension join has duplicated a line. |
| `PCT_DATE_OK` | **99.66%** | The 0.34% shortfall is the unparseable legacy dates, now flagged rather than lost. |
| `PCT_ENTITY_OK` | **100.00%** | Every line resolved to a group entity through the crosswalk. |
| `PCT_ACCOUNT_OK` | **96.77%** | 3.23% of the ledger carries an account code nobody has mapped. |
| `PCT_SUPPLIER_OK` | **100.00%** | Every line resolved to a supplier record. |

> ⚠️ **The fan-out check is not a formality**
>
> `COUNT(*) = COUNT(DISTINCT GL_LINE_ID)` is the assertion that no join multiplied a ledger line. If a dimension table ever gains a duplicate key, this is the number that moves — and it moves before anyone notices that the GST total has grown.

### Block 4 — inspect the object you created

In [ ]:
SHOW DYNAMIC TABLES IN SCHEMA CURATED;

Read across the row: the target lag you declared, the warehouse that maintains it, the scheduling state and the refresh mode Snowflake actually chose. You asked for `REFRESH_MODE = AUTO`, which means the platform decides how to keep the table current rather than you deciding for it — so read what it reports rather than assuming a value. `TARGET_LAG` is a freshness objective, not a schedule: it is a promise about how stale the data may become, not an instruction to run at a fixed time.

> ✅ **Where you are**
>
> One governed ledger, 2,400,000 lines, three source systems, four dimensions resolved and four data-quality conditions labelled. It contains no tax logic whatsoever — and that is what makes the next step possible.

---

## Step 4 — The tax rules layer

*~15 min* · [narrative & expected results](gsi_tax_workshop_lab1_foundation.html#step-4)

This is the most important step in the lab, and it carries a single message: **the rules live in a table that the tax team owns, not in code that a developer owns.**

Tax logic is not a `CASE` statement. It is a body of rules with jurisdictions, effective dates and precedence, and it changes without warning — usually at short notice, and usually not by anyone in IT. So we hold it as data the tax team can read, review and change themselves.

- Read the entire determination logic of the system without reading any SQL.
- Apply effective-dated rules with generic SQL that contains no rates and no treatments.
- Watch the group GST position change from a single `UPDATE` to one row of data.
- Stand up a self-maintaining exception register over the result.

### Block 1 — the rules, as data

In [ ]:
-- The decision table. NULL means "matches anything". Lower PRIORITY wins.
SELECT RULE_ID, PRIORITY, JURISDICTION, SUPPLY_CATEGORY,
       SUPPLIER_GST_REGISTERED, IS_CROSS_BORDER,
       TAX_TREATMENT, RECOVERABLE,
       VALID_FROM, VALID_TO, RULE_DESCRIPTION
FROM TAX.TAX_RULE_DETERMINATION
ORDER BY PRIORITY, RULE_ID;

In [ ]:
-- The rate table. Keyed on jurisdiction, treatment and date.
SELECT RATE_ID, JURISDICTION, TAX_TREATMENT, RATE_PCT, VALID_FROM, VALID_TO
FROM TAX.TAX_RULE_RATE
ORDER BY JURISDICTION, VALID_FROM, TAX_TREATMENT;

Stop here and say this out loud, because it is the point of the step: **a tax person can read those two result grids and understand the entire determination logic of the system without reading a single line of SQL.** Sixteen rules and nineteen rates. No code review, no developer to interpret, nothing hidden in a query somebody wrote four years ago and left.

Now let the room find **rules 9 and 16**. Same priority, same predicates — `PROFESSIONAL_FEES`, supplier not GST registered, cross-border — and two adjacent, non-overlapping validity windows:

| Rule | Priority | Predicates | Treatment | Recoverable | Valid from | Valid to |
|---|---|---|---|---|---|---|
| **9** | 20 | `PROFESSIONAL_FEES`, unregistered, cross-border | **GST_FREE** | FALSE | 2000-07-01 | **2024-06-30** |
| **16** | 20 | `PROFESSIONAL_FEES`, unregistered, cross-border | **REVERSE_CHARGE** | TRUE | **2024-07-01** | 9999-12-31 |

That is not two rules. It is **the same rule with two effective-dated versions**. Following a review, the group corrected its position on imported professional services with effect from 1 July 2024 — before that date those acquisitions were treated as out of scope, and from that date the reverse charge applies. Unrecognised reverse charge on imported services is one of the most common findings in an Australian GST review, so this is not a hypothetical.

> ⚠️ **Ask the room this question before you run anything else**
>
> What happens to three years of already-lodged BAS if that corrected treatment is *hard-coded* instead of effective-dated? You do not merely get next quarter wrong.

One note on the rate table while it is on screen. The Australian GST rate has been 10% since 1 July 2000 and has never changed — so the demonstrated change in this workshop is a change of *treatment*, not of rate, because that is what actually moves in Australia. The rate table is date-effective anyway, so that if the rate ever does move, nobody edits a query. Its non-Australian rows exist only as unused extensibility and no example in this workshop touches them.

Rule 15 deserves one sentence too. It matches everything at priority 99, which guarantees that every line resolves to exactly one rule. A decision table without a catch-all silently drops rows, and dropped rows in a tax computation are the worst possible failure mode because the total still looks reasonable.

### Why two rule tables and not one

Different grain. Determination is keyed on the characteristics of the transaction — jurisdiction, supply category, whether the supplier is registered, whether the supply is cross-border. Rate is keyed on jurisdiction, treatment and date. Collapse them into one table and a single rate change becomes an update across every rule that happens to mention that treatment, which is exactly the kind of multi-row edit that gets one row wrong.

> 💡 **And why the rules are not SQL strings in a VARCHAR**
>
> A recurring suggestion is to store each rule as a SQL fragment in a text column and evaluate it dynamically. Do not: a rule expressed as a string cannot be tested, validated on entry, or type-checked, so the first time you learn it is malformed is when it runs in a production computation.

### Block 2 — build the determination view

In [ ]:
CREATE OR REPLACE VIEW TAX.V_GST_DETERMINED AS
WITH matched AS (
    SELECT
        g.*,
        r.RULE_ID          AS APPLIED_RULE_ID,
        r.RULE_DESCRIPTION AS APPLIED_RULE_DESCRIPTION,
        r.TAX_TREATMENT,
        r.RECOVERABLE
    FROM CURATED.GL_HARMONISED g
    LEFT JOIN TAX.TAX_RULE_DETERMINATION r
      ON (r.JURISDICTION            IS NULL OR r.JURISDICTION            = g.JURISDICTION)
     AND (r.SUPPLY_CATEGORY         IS NULL OR r.SUPPLY_CATEGORY         = g.SUPPLY_CATEGORY)
     AND (r.SUPPLIER_GST_REGISTERED IS NULL OR r.SUPPLIER_GST_REGISTERED = g.SUPPLIER_GST_REGISTERED)
     AND (r.IS_CROSS_BORDER         IS NULL OR r.IS_CROSS_BORDER         = g.IS_CROSS_BORDER)
     AND  g.DOC_DATE BETWEEN r.VALID_FROM AND r.VALID_TO
    QUALIFY ROW_NUMBER() OVER (PARTITION BY g.GL_LINE_ID
                               ORDER BY r.PRIORITY, r.RULE_ID) = 1
)
SELECT
    m.*,
    rt.RATE_PCT                                          AS APPLIED_RATE_PCT,
    ROUND(m.NET_AMOUNT * rt.RATE_PCT / 100, 2)           AS EXPECTED_TAX_AMOUNT,
    ROUND(m.BOOKED_TAX_AMOUNT
          - ROUND(m.NET_AMOUNT * rt.RATE_PCT / 100, 2), 2) AS TAX_VARIANCE
FROM matched m
LEFT JOIN TAX.TAX_RULE_RATE rt
       ON rt.JURISDICTION   = m.JURISDICTION
      AND rt.TAX_TREATMENT  = m.TAX_TREATMENT
      AND m.DOC_DATE BETWEEN rt.VALID_FROM AND rt.VALID_TO;

Read what is *not* in that statement. There are no rates, no jurisdictions, no supply categories and no treatments anywhere in it. It is a thin, generic mechanism for applying whatever the two tables happen to say. Change a rule and you have changed the answer this view returns on the next query — no rebuild, no redeployment.

Both joins are date-ranged on `DOC_DATE` and not on `CURRENT_DATE()`. This is the whole idea: a transaction is judged by the rules that applied *when it happened*. It is also why rules 9 and 16 can share priority 20 without ambiguity — their windows do not overlap, so only one of them can ever match a given line.

> 💡 **QUALIFY, in one paragraph**
>
> Most attendees will not have met `QUALIFY`. It filters on the result of a window function, in the same way `HAVING` filters on the result of an aggregate.

The join to the rule table is a `LEFT JOIN`, deliberately. A line whose date could not be parsed has no determinable treatment — you genuinely cannot say how a transaction should be taxed when you do not know when it occurred. With an inner join those lines vanish silently, which is catastrophic, because they are precisely the lines somebody needs to chase. As a left join all 2,400,000 lines survive and `TAX_TREATMENT` comes back `NULL` only where `IS_DATE_PARSED = FALSE`. Missing information is represented as missing information.

### Block 3 — prove the effective dating

One query, one population: Australian imported professional fees from offshore suppliers that are not GST registered. Grouped either side of 1 July 2024, from the same view, with no conditional tax logic anywhere.

In [ ]:
SELECT CASE WHEN DOC_DATE < '2024-07-01'::DATE
            THEN 'Before 1 Jul 2024'
            ELSE 'From 1 Jul 2024' END      AS PERIOD_SIDE,
       APPLIED_RULE_ID,
       TAX_TREATMENT,
       APPLIED_RATE_PCT,
       COUNT(*)                             AS LINE_COUNT,
       ROUND(SUM(EXPECTED_TAX_AMOUNT), 2)   AS EXPECTED_GST
FROM TAX.V_GST_DETERMINED
WHERE JURISDICTION            = 'AU'
  AND SUPPLY_CATEGORY         = 'PROFESSIONAL_FEES'
  AND SUPPLIER_GST_REGISTERED = FALSE
  AND IS_CROSS_BORDER         = TRUE
  AND DOC_DATE IS NOT NULL
GROUP BY 1, 2, 3, 4
ORDER BY 1;

| Period | Rule | Treatment | Rate | Lines | Expected GST |
|---|---|---|---|---|---|
| Before 1 Jul 2024 | **9** | `GST_FREE` | 0.00% | 2,429 | **$0** |
| From 1 Jul 2024 | **16** | `REVERSE_CHARGE` | 10.00% | 4,690 | **$26,247,261** |

An otherwise identical population, split only by document date, produces two different treatments and **$25.5 million** of expected GST on one side of the line that does not exist on the other. Nothing in the query decided that. The rows in the rule table decided it.

> ⚠️ **Now make the consequence explicit**
>
> Had the corrected treatment been hard-coded — a `CASE` expression saying imported professional fees are reverse charged — the *before* row above would read `REVERSE_CHARGE` at 10.00% as well. Every period already lodged on the old position would be silently restated, and the reconciliation in Step 5 would raise variances against returns that were correct when they were filed.

### Block 4 — the moment

Someone in the room is about to ask what happens when the tax team wants a change. Answer it by doing it. Run all three statements in order.

In [ ]:
-- 1. The tax team wants to model the reverse charge at a different rate.
--    They change one row of data. Not a query. Not a deployment.
UPDATE TAX.TAX_RULE_RATE
   SET RATE_PCT = 15.00
 WHERE JURISDICTION = 'AU' AND TAX_TREATMENT = 'REVERSE_CHARGE';

In [ ]:
-- 2. Exactly the same aggregate as Block 3. Nothing was rebuilt,
--    nothing was refreshed, nothing was redeployed.
SELECT APPLIED_RULE_ID,
       TAX_TREATMENT,
       APPLIED_RATE_PCT,
       COUNT(*)                             AS LINE_COUNT,
       ROUND(SUM(EXPECTED_TAX_AMOUNT), 2)   AS EXPECTED_GST
FROM TAX.V_GST_DETERMINED
WHERE JURISDICTION            = 'AU'
  AND SUPPLY_CATEGORY         = 'PROFESSIONAL_FEES'
  AND SUPPLIER_GST_REGISTERED = FALSE
  AND IS_CROSS_BORDER         = TRUE
  AND DOC_DATE >= '2024-07-01'::DATE
GROUP BY 1, 2, 3;

In [ ]:
-- 3. Put it back. The Australian rate is 10% and has been since 2000.
UPDATE TAX.TAX_RULE_RATE
   SET RATE_PCT = 10.00
 WHERE JURISDICTION = 'AU' AND TAX_TREATMENT = 'REVERSE_CHARGE';

`APPLIED_RATE_PCT` now reads 15.00 and the expected GST on those 4,690 lines has risen by half, to roughly $38.3 million. Then statement 3 puts it back and the position returns to $26,247,261. The group's computed GST position moved twice in about four seconds.

> ✅ **This is ownership, not architecture**
>
> No code was edited. Nothing was rebuilt, nothing was redeployed, no release was cut and no developer was involved.

### Block 5 — the standing exception register

One view that turns the determined ledger into a work queue: every line that fails a tax test, with the reasons concatenated and a risk score to sort by.

In [ ]:
CREATE OR REPLACE VIEW TAX.V_GST_EXCEPTIONS AS
WITH d AS (SELECT * FROM TAX.V_GST_DETERMINED),
ex AS (
    SELECT GL_LINE_ID, 'E1' AS EXCEPTION_CODE, 'MEDIUM' AS SEVERITY,
           'Account code ' || COALESCE(NATIVE_ACCOUNT_CODE,'?') || ' has no group mapping' AS EXCEPTION_REASON
    FROM d WHERE NOT IS_ACCOUNT_MAPPED
    UNION ALL
    SELECT GL_LINE_ID, 'E2', 'HIGH', 'Document date could not be parsed from the source system'
    FROM d WHERE NOT IS_DATE_PARSED   -- reachable only because of the LEFT JOIN in V_GST_DETERMINED
    UNION ALL
    SELECT GL_LINE_ID, 'E3', 'HIGH', 'Supplier ABN missing on a claimed credit over $82.50'
    FROM d WHERE SUPPLIER_ABN IS NULL AND DOC_TYPE = 'AP_INVOICE'
             AND RECOVERABLE AND ABS(GROSS_AMOUNT) > 82.50
    UNION ALL
    SELECT GL_LINE_ID, 'E4', 'HIGH', 'GST charged by a supplier that is not GST registered'
    FROM d WHERE SUPPLIER_GST_REGISTERED = FALSE AND BOOKED_TAX_AMOUNT <> 0
    UNION ALL
    SELECT GL_LINE_ID, 'E5', 'MEDIUM',
           'Booked GST differs from expected by ' || TO_VARCHAR(ROUND(TAX_VARIANCE,2))
    FROM d WHERE ABS(COALESCE(TAX_VARIANCE, 0)) > 1.00
    UNION ALL
    SELECT GL_LINE_ID, 'E6', 'HIGH',
           'Credit claimed on a non-recoverable treatment (' || TAX_TREATMENT || ')'
    FROM d WHERE RECOVERABLE = FALSE AND TAX_TREATMENT <> 'NON_DEDUCTIBLE'
             AND DOC_TYPE = 'AP_INVOICE' AND BOOKED_TAX_AMOUNT <> 0
    UNION ALL
    SELECT GL_LINE_ID, 'E7', 'MEDIUM', 'Reverse charge required but not applied in the source system'
    FROM d WHERE TAX_TREATMENT = 'REVERSE_CHARGE'
             AND COALESCE(BOOKED_TAX_CODE,'') NOT IN ('RC','REVERSE_CHARGE')
)
SELECT
    d.GL_LINE_ID, d.ENTITY_ID, d.ENTITY_NAME, d.TAX_PERIOD,
    d.SUPPLIER_ID, d.SUPPLIER_NAME,
    d.SOURCE_SYSTEM, d.SOURCE_DOC_REF, d.DOC_TYPE, d.SUPPLY_CATEGORY,
    d.NET_AMOUNT, d.BOOKED_TAX_AMOUNT, d.EXPECTED_TAX_AMOUNT, d.TAX_VARIANCE,
    d.TAX_TREATMENT, d.APPLIED_RULE_ID,
    COUNT(*)                                                        AS EXCEPTION_COUNT,
    LISTAGG(DISTINCT ex.EXCEPTION_CODE, ', ')
        WITHIN GROUP (ORDER BY ex.EXCEPTION_CODE)                   AS EXCEPTION_CODES,
    LISTAGG(ex.EXCEPTION_REASON, ' | ')
        WITHIN GROUP (ORDER BY ex.EXCEPTION_CODE)                   AS EXCEPTION_REASONS,
    CASE MAX(CASE ex.SEVERITY WHEN 'HIGH' THEN 3 WHEN 'MEDIUM' THEN 2 ELSE 1 END)
         WHEN 3 THEN 'HIGH' WHEN 2 THEN 'MEDIUM' ELSE 'LOW' END     AS MAX_SEVERITY,
    ROUND(COUNT(*) * 10
          + LEAST(ABS(COALESCE(d.TAX_VARIANCE, 0)) / 100, 50), 2)   AS RISK_SCORE
FROM ex
JOIN d ON d.GL_LINE_ID = ex.GL_LINE_ID
GROUP BY ALL;

In [ ]:
-- The work queue: worst first.
SELECT GL_LINE_ID, ENTITY_NAME, TAX_PERIOD, SUPPLIER_NAME,
       SUPPLY_CATEGORY, TAX_TREATMENT, APPLIED_RULE_ID,
       NET_AMOUNT, BOOKED_TAX_AMOUNT, EXPECTED_TAX_AMOUNT, TAX_VARIANCE,
       EXCEPTION_COUNT, EXCEPTION_CODES, MAX_SEVERITY, RISK_SCORE
FROM TAX.V_GST_EXCEPTIONS
ORDER BY RISK_SCORE DESC
LIMIT 50;

That is a standing exception register: seven tax tests over the whole ledger, always current, always sorted worst-first.

| Code | Test | Lines |
|---|---|---|
| `E1` | Account code has no group mapping | 77,484 |
| `E2` | Document date could not be parsed | 8,088 |
| `E3` | Supplier ABN missing on a claimed credit over $82.50 | 60,097 |
| `E4` | GST charged by a supplier that is not registered | 31,039 |
| `E5` | Booked GST differs from expected by more than $1 | 107,808 |
| `E6` | Credit claimed on a non-recoverable treatment | 36,718 |
| `E7` | Reverse charge required but not applied at source | 11,754 |
| **Distinct ledger lines flagged** |  | **253,394 — 10.56% of the ledger** |

A line can fail more than one test, which is why the code counts sum to more than 253,394 and why `EXCEPTION_COUNT` feeds the risk score. Note that `E2` returns exactly the 8,088 unparseable dates from Step 3 — the left join in Block 2 is what allowed those lines to reach this register at all. Lab 2 comes back to this queue and uses Cortex to explain each flagged line in plain English.

> ✅ **What you just built**
>
> Sixteen determination rules and nineteen rates, held as data the tax team owns and can change without a developer; a thin generic view that applies whichever rules were in force on the date of each transaction; and a standing exception register over 2,400,000 ledger lines.

### This is one rule set, not *the* rule set

What you built is not "the tax rules". It is **one rule set, for one tax**, sitting beside a ledger that knows nothing about GST — which is exactly why tax logic was kept out of the curated layer in Step 3.

**The ledger is bespoke to the client. The rule sets are not.** The payroll tax rule set you build for one client is most of the payroll tax rule set for the next one.

> ✅ **The commercial point**
>
> Most of this work today is billed by the hour and thrown away when the engagement closes. Build it this way and the rule libraries become your firm's own intellectual property — something you deploy repeatedly rather than rebuild from scratch for every client.

> ⚠️ **The honest boundary**
>
> Not everything is a rule table. Anything requiring iterative or circular computation needs procedural logic — thin capitalisation, Pillar Two top-up calculations, and consolidation eliminations with circular ownership are all in that category.

---

## Step 5 — Reconcile, then survive the audit

*~15 min* · [narrative & expected results](gsi_tax_workshop_lab1_foundation.html#step-5)

You now have a computed GST position for every entity and every period. The client has a lodged one, sitting in `RAW_DATA.FCT_BAS_LODGED` — 532 rows, fourteen entities across thirty-eight complete months. Nobody has ever put the two side by side, because until this morning there was no computed position to compare against.

Then, halfway through the reconciliation, someone posts a backdated adjustment into the ledger — and the auditor asks the question every tax practitioner dreads: *show me the ledger exactly as it stood when we lodged.*

- Reconcile computed GST against lodged BAS at entity and period, and classify the result.
- Watch a backdated posting break a reconciliation that tied five minutes ago.
- Recover the as-lodged position with one clause of SQL — no backup, no restore.
- Take an immutable audit snapshot of a 2,400,000-row ledger, instantly and for no storage.
- Undo a `DROP TABLE` you should not have run.

### Block 1 — build the reconciliation and run it

The view below is deliberately conservative in three places. It restricts to `JURISDICTION = 'AU'`, because the lodged data is Australian BAS. It excludes the current, incomplete month, which has no lodged BAS yet and would otherwise appear as a large false variance every time the workshop is run. And it uses a `FULL OUTER JOIN`, so a period that exists on one side only still appears rather than vanishing — a lodged period with no computed lines is exactly as interesting as the reverse.

In [ ]:
CREATE OR REPLACE VIEW TAX.V_GST_VS_BAS AS
WITH computed AS (
    SELECT ENTITY_ID, ENTITY_NAME, TAX_PERIOD,
           ABS(COALESCE(SUM(CASE WHEN DOC_TYPE='AR_INVOICE'
                                 THEN EXPECTED_TAX_AMOUNT END), 0)) AS COMPUTED_1A,
           ABS(COALESCE(SUM(CASE WHEN DOC_TYPE='AP_INVOICE' AND RECOVERABLE
                                 THEN EXPECTED_TAX_AMOUNT END), 0)) AS COMPUTED_1B
    FROM TAX.V_GST_DETERMINED
    WHERE JURISDICTION = 'AU' AND TAX_PERIOD IS NOT NULL
      -- exclude the current, incomplete month: it has no lodged BAS yet and
      -- would otherwise show as a false variance
      AND TAX_PERIOD < DATE_TRUNC('month', '2026-07-31'::DATE)
    GROUP BY ENTITY_ID, ENTITY_NAME, TAX_PERIOD
)
SELECT
    COALESCE(c.ENTITY_ID, b.ENTITY_ID)                              AS ENTITY_ID,
    COALESCE(c.ENTITY_NAME, e.ENTITY_NAME)                          AS ENTITY_NAME,
    COALESCE(c.TAX_PERIOD, b.TAX_PERIOD)                            AS TAX_PERIOD,
    ROUND(COALESCE(c.COMPUTED_1A, 0), 2)                            AS COMPUTED_1A,
    ROUND(COALESCE(c.COMPUTED_1B, 0), 2)                            AS COMPUTED_1B,
    ROUND(COALESCE(c.COMPUTED_1A, 0) - COALESCE(c.COMPUTED_1B, 0), 2) AS COMPUTED_NET_GST,
    COALESCE(b.LABEL_1A_GST_ON_SALES, 0)                            AS LODGED_1A,
    COALESCE(b.LABEL_1B_GST_CREDITS, 0)                             AS LODGED_1B,
    COALESCE(b.NET_GST, 0)                                          AS LODGED_NET_GST,
    ROUND(COALESCE(c.COMPUTED_1A,0) - COALESCE(b.LABEL_1A_GST_ON_SALES,0), 2) AS VARIANCE_1A,
    ROUND(COALESCE(c.COMPUTED_1B,0) - COALESCE(b.LABEL_1B_GST_CREDITS,0), 2)  AS VARIANCE_1B,
    ROUND((COALESCE(c.COMPUTED_1A,0) - COALESCE(c.COMPUTED_1B,0))
          - COALESCE(b.NET_GST,0), 2)                               AS VARIANCE_NET,
    ROUND(((COALESCE(c.COMPUTED_1A,0) - COALESCE(c.COMPUTED_1B,0)) - COALESCE(b.NET_GST,0))
          / NULLIF(ABS(COALESCE(b.NET_GST,0)), 0) * 100, 2)         AS VARIANCE_PCT,
    CASE
      WHEN ABS((COALESCE(c.COMPUTED_1A,0)-COALESCE(c.COMPUTED_1B,0))
               - COALESCE(b.NET_GST,0)) <= 100  THEN 'MATCHED'
      WHEN ABS((COALESCE(c.COMPUTED_1A,0)-COALESCE(c.COMPUTED_1B,0))
               - COALESCE(b.NET_GST,0)) <= 1000 THEN 'REVIEW'
      ELSE 'INVESTIGATE'
    END                                                             AS RECON_STATUS
FROM computed c
FULL OUTER JOIN RAW_DATA.FCT_BAS_LODGED b
     ON b.ENTITY_ID = c.ENTITY_ID AND b.TAX_PERIOD = c.TAX_PERIOD
LEFT JOIN RAW_DATA.DIM_LEGAL_ENTITY e
     ON e.ENTITY_ID = COALESCE(c.ENTITY_ID, b.ENTITY_ID);

In [ ]:
-- The reconciliation. One row per entity-period, worst first.
SELECT RECON_STATUS,
       COUNT(*)                        AS ENTITY_PERIODS,
       ROUND(SUM(ABS(VARIANCE_NET)),2) AS TOTAL_ABS_VARIANCE
FROM TAX.V_GST_VS_BAS
GROUP BY RECON_STATUS
ORDER BY TOTAL_ABS_VARIANCE DESC;

| Recon status | Entity-periods | What it means |
|---|---|---|
| `MATCHED` | **514** | Variance of exactly **$0.00**. Not "within tolerance" — nil. |
| `INVESTIGATE` | **18** | Variance above $1,000. These are the conversation. |
| **532 entity-periods** |  | Fourteen entities across thirty-eight complete months. |

Read the 514 first. Two independently produced numbers — one computed from 2,400,000 ledger lines through a rule set the tax team owns, one lodged with the ATO months or years ago — agree to the cent for 514 of 532 entity-periods. That is the evidence that the pipeline is right, and it is what makes the remaining 18 credible. A reconciliation that flags everything tells you nothing; this one flags 3.4% of periods and can defend the other 96.6%.

> 💡 **This is the moment the pipeline pays for itself**
>
> Every step so far was investment: conforming three ERPs, resolving entities, externalising the rules. This single query is the return.

### Block 2 — the adjustment lands

Now the realistic complication. Somebody in the business posts a backdated journal against a period that was lodged six months ago. They are not doing anything wrong: late invoices, accruals reversed, an intercompany correction. It happens constantly, and the tax team almost never hears about it.

In [ ]:
-- 1. Mark the moment. Everything before this timestamp is the ledger as it
--    stood when the BAS was lodged. This is what makes the next step work:
--    you are not guessing how long ago something happened, you named it.
SET pre_adjustment = (SELECT CURRENT_TIMESTAMP());

In [ ]:
-- 2. A backdated AR adjustment against ENT001 (SAP company code 1010),
--    dated into the month that closed six months ago. AR invoices are
--    booked negative in SAP, so both amounts carry a minus sign.
INSERT INTO RAW_DATA.ERP_SAP_GL
    (BELNR, BUZEI, BUKRS, HKONT, BUDAT, DMBTR, MWSTS, WAERS, MWSKZ, LIFNR, DOC_TYPE, SGTXT)
SELECT 'SAP9900001', 1, '1010', '400001',
       TO_VARCHAR(DATEADD('month', -6, DATE_TRUNC('month', '2026-07-31'::DATE)), 'YYYYMMDD'),
       -1500000.00, -150000.00, 'AUD', 'S1', 'SUP000001', 'AR_INVOICE',
       'Backdated revenue adjustment - posted after BAS lodgement';

In [ ]:
-- 3. Force the Dynamic Table to pick it up now rather than waiting for
--    its one-hour target lag. TARGET_LAG is a freshness objective;
--    this is how you demand freshness immediately.
ALTER DYNAMIC TABLE CURATED.GL_HARMONISED REFRESH;

In [ ]:
-- 4. Re-run the reconciliation for that entity and period only.
SELECT ENTITY_NAME, TAX_PERIOD, COMPUTED_1A, LODGED_1A,
       VARIANCE_NET, VARIANCE_PCT, RECON_STATUS
FROM TAX.V_GST_VS_BAS
WHERE ENTITY_ID  = 'ENT001'
  AND TAX_PERIOD = DATEADD('month', -6, DATE_TRUNC('month', '2026-07-31'::DATE));

That entity-period was `MATCHED` at $0.00 four minutes ago. It is now `INVESTIGATE`, carrying roughly $150,000 of GST on sales that the lodged BAS does not contain. Nothing was recalculated by hand and no report was regenerated: one row landed in a raw table, the Dynamic Table refreshed, and the reconciliation told a different and correct story.

> ⚠️ **The lodged figure has not changed — and cannot**
>
> This is the distinction that matters to a tax practitioner. The BAS you lodged is a historical fact.

### Block 3 — the as-lodged position, in one clause

The auditor's question is now live: *show me the ledger exactly as it stood when we lodged.* The answer is not a backup, a restore, a snapshot table somebody remembered to populate, or a ticket to IT. It is a clause.

In [ ]:
SELECT COUNT(*)                    AS LINES_AS_AT_THAT_MOMENT,
       ROUND(SUM(BOOKED_TAX_AMOUNT), 2) AS BOOKED_TAX_AS_LODGED
FROM CURATED.GL_HARMONISED AT(TIMESTAMP => $pre_adjustment)
WHERE ENTITY_ID  = 'ENT001'
  AND TAX_PERIOD = DATEADD('month', -6, DATE_TRUNC('month', '2026-07-31'::DATE));

That is the ledger as it stood at the moment you named, before the adjustment existed. The row count is one lower and the booked tax total is the as-lodged figure — the number that ties to the BAS.

Notice what you did **not** do: you did not restore a database, you did not stand up a copy, you did not go looking for last quarter's extract, and you did not ask anyone's permission. You added eighteen characters to a query you already had.

### Block 4 — freeze the evidence, then break it and get it back

Time Travel answers the question in the moment. The auditor wants something they can come back to next month, so freeze it: a clone of the ledger as it stood before the adjustment, named for the date it was taken, in a schema that exists for exactly this purpose.

In [ ]:
-- An immutable, named snapshot of the whole 2,400,000-line ledger
-- as it stood at the moment before the adjustment was posted.
CREATE TABLE AUDIT.GL_AS_LODGED_20260803 CLONE CURATED.GL_HARMONISED AT(TIMESTAMP => $pre_adjustment);

In [ ]:
SELECT COUNT(*) AS SNAPSHOT_LINES FROM AUDIT.GL_AS_LODGED_20260803;

In [ ]:
-- Now the accident everybody has had. Wrong window, wrong object.
DROP TABLE AUDIT.GL_AS_LODGED_20260803;

In [ ]:
-- And the recovery. Within the retention window, one word.
UNDROP TABLE AUDIT.GL_AS_LODGED_20260803;

In [ ]:
SELECT COUNT(*) AS STILL_HERE FROM AUDIT.GL_AS_LODGED_20260803;

> ✅ **Zero-copy cloning — what just happened, and what it cost**
>
> You took an immutable snapshot of a **2,400,000-row ledger**. It completed instantly and it consumed **no storage**, because a clone copies metadata rather than data — the two objects share the same underlying micro-partitions until one of them diverges, and you pay only for what subsequently changes.

The wider point is that none of this required a backup strategy, a restore procedure, or a conversation with IT. There was no data protection design decision to make in advance, no retention policy to negotiate, and no recovery runbook to test. Time Travel, cloning and `UNDROP` are properties of the platform — they are true of every table in the account, whether or not anybody planned for them.

### Block 5 — the visual close, with no new SQL

Re-run the reconciliation, then read it as a picture rather than a grid. There is no charting library to install and no new query to write.

In [ ]:
SELECT ENTITY_NAME,
       ROUND(SUM(ABS(VARIANCE_NET)), 2) AS TOTAL_ABS_VARIANCE
FROM TAX.V_GST_VS_BAS
GROUP BY ENTITY_NAME
ORDER BY TOTAL_ABS_VARIANCE DESC;

1. Run the query and wait for the grid to appear.
2. Click the **Chart** tab, immediately beside **Results** above the output pane.
3. Set the chart type to **Bar** in the panel on the right.
4. Set the X axis to `ENTITY_NAME` and the Y axis to `TOTAL_ABS_VARIANCE`.

The distribution is the finding. Variance is not spread evenly across the group — it concentrates in a handful of entities, and those are where the tax team should spend its week. This is the chart you put in front of the client, built from a query you were running anyway, and it takes four clicks.

> 💡 **Why this matters more than it looks**
>
> Most analysis dies between the query and the audience. A number in a results grid is not a finding until somebody can see the shape of it.

### What you built

| Object | Type | Rows | Purpose |
|---|---|---|---|
| `RAW_DATA.ERP_SAP_GL` | TABLE | 1,200,000 | SAP general ledger, as landed. |
| `RAW_DATA.ERP_ORACLE_GL` | TABLE | 800,000 | Oracle general ledger, as landed. |
| `RAW_DATA.ERP_LEGACY_AP` | TABLE | 400,000 | Legacy AP system, text dates, GST-inclusive amounts. |
| `RAW_DATA.DIM_LEGAL_ENTITY` | TABLE | 14 | Group entities and the three ERP identifier schemes. |
| `RAW_DATA.DIM_ACCOUNT_MAPPING` | TABLE | 180 | Native account code to group account and supply category. |
| `RAW_DATA.DIM_SUPPLIER` | TABLE | 2,000 | Supplier ABN, GST registration and country. |
| `RAW_DATA.FCT_BAS_LODGED` | TABLE | 532 | What was actually lodged with the ATO. |
| `RAW_DATA.DOC_TAX_INVOICE` | TABLE | 300 | Unstructured invoice text — Lab 2 uses this. |
| `TAX.TAX_RULE_DETERMINATION` | TABLE | 16 | The decision table: *the first rule set of many — see the end of Step 4*. |
| `TAX.TAX_RULE_RATE` | TABLE | 19 | Effective-dated rates, part of the same rules layer. |
| `CURATED.GL_HARMONISED` | **DYNAMIC TABLE** | 2,400,000 | One governed ledger. No tax logic. Keeps itself current. |
| `TAX.V_GST_DETERMINED` | VIEW | 2,400,000 | Treatment, rate and variance per line, with rule lineage. |
| `TAX.V_GST_VS_BAS` | VIEW | 532 | Computed against lodged, by entity and period. |
| `TAX.V_GST_EXCEPTIONS` | VIEW | 253,394 | Risk-ranked exception register, seven rules. |

Four objects of your own creation on top of the source data, one of which is a Dynamic Table and three of which are views — and a reconciliation that ties to the cent for 514 of 532 entity-periods. There are three optional steps below: a Streamlit control tower over these two views, a scenario-modelling exercise on a cloned database, and a secure share of the audit snapshot. Lab 2 takes the same objects into Cortex AI — classification, extraction and a Cortex Agent your client can ask questions of in English.

> ⚠️ **There is deliberately no cleanup in Lab 1**
>
> Do not drop `GSI_TAX_WORKSHOP` or any object inside it. Lab 2 builds directly on `CURATED.GL_HARMONISED`, `TAX.V_GST_DETERMINED`, `TAX.V_GST_VS_BAS` and `TAX.V_GST_EXCEPTIONS`.

---

## Step 6 — Streamlit tax control tower

*~12 min · OPTIONAL* · [narrative & expected results](gsi_tax_workshop_lab1_foundation.html#step-6)

**⭐ RECOMMENDED FIRST.** Of the three optional steps this is the one to do if you only do one. Everything you have built so far lives in a results grid; this turns it into an application your client can open, filter and use — written in Python, running inside Snowflake, with no infrastructure, no deployment pipeline and no connection string.

- Build a three-tab tax control tower over `TAX.V_GST_VS_BAS` and `TAX.V_GST_EXCEPTIONS`.
- Use the session that already exists rather than authenticating again.
- Cache every query so that filtering the page does not re-scan the ledger.
- Understand why every query in the app aggregates or limits before it returns.

### Create the app in Snowsight

1. In the left navigation choose **Projects**, then **Streamlit**.
2. Click **+ Streamlit App**.
3. Name it `GST Tax Control Tower`.
4. Set **App location** to database `GSI_TAX_WORKSHOP` and schema `APPS`.
5. Set **App warehouse** to `GSI_TAX_WH`.
6. Click **Create**, delete the sample code in the editor, paste the block below, and click **Run**.

The app runs unmodified. There is nothing to edit, nothing to configure and no packages to add — Streamlit and the Snowpark library are already present in the environment.

### Block 1 — the application

In [ ]:
# Streamlit app code. Run this cell to preview the app inline, or paste it
# into Projects -> Streamlit to deploy it standalone (add st.set_page_config
# back at the top when you do).
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

# The session already exists. Never build a connection inside
# Streamlit in Snowflake.
session = get_active_session()


# ---------------------------------------------------------------------
# Every query below is aggregated or LIMITed server side. The ledger is
# 2,400,000 lines and the exception register is 253,394 - returning either
# of them row level would time the app out and achieve nothing, because
# nobody reads 253,394 rows. Aggregate in Snowflake, render the summary.
# ---------------------------------------------------------------------

@st.cache_data(ttl=600)
def load_headline():
    return session.sql("""
        SELECT COUNT(*)                                            AS ENTITY_PERIODS,
               SUM(IFF(RECON_STATUS = 'MATCHED', 1, 0))            AS MATCHED,
               SUM(IFF(RECON_STATUS = 'INVESTIGATE', 1, 0))        AS INVESTIGATE,
               ROUND(SUM(ABS(VARIANCE_NET)), 2)                    AS TOTAL_ABS_VARIANCE
        FROM TAX.V_GST_VS_BAS
    """).to_pandas()


@st.cache_data(ttl=600)
def load_exception_headline():
    return session.sql("""
        SELECT COUNT(*)                                    AS FLAGGED_LINES,
               SUM(IFF(MAX_SEVERITY = 'HIGH', 1, 0))       AS HIGH_SEVERITY,
               ROUND(AVG(RISK_SCORE), 2)                   AS AVG_RISK_SCORE
        FROM TAX.V_GST_EXCEPTIONS
    """).to_pandas()


@st.cache_data(ttl=600)
def load_recon(status_filter):
    where = ""
    if status_filter != "All":
        where = f"WHERE RECON_STATUS = '{status_filter}'"
    return session.sql(f"""
        SELECT ENTITY_NAME, TAX_PERIOD, COMPUTED_NET_GST, LODGED_NET_GST,
               VARIANCE_NET, VARIANCE_PCT, RECON_STATUS
        FROM TAX.V_GST_VS_BAS
        {where}
        ORDER BY ABS(VARIANCE_NET) DESC
        LIMIT 500
    """).to_pandas()


@st.cache_data(ttl=600)
def load_variance_by_entity():
    return session.sql("""
        SELECT ENTITY_NAME,
               ROUND(SUM(ABS(VARIANCE_NET)), 2) AS TOTAL_ABS_VARIANCE
        FROM TAX.V_GST_VS_BAS
        GROUP BY ENTITY_NAME
        ORDER BY TOTAL_ABS_VARIANCE DESC
    """).to_pandas()


@st.cache_data(ttl=600)
def load_filter_values():
    entities = session.sql("""
        SELECT DISTINCT ENTITY_NAME FROM TAX.V_GST_EXCEPTIONS ORDER BY 1
    """).to_pandas()["ENTITY_NAME"].tolist()
    codes = session.sql("""
        SELECT DISTINCT VALUE::VARCHAR AS EXCEPTION_CODE
        FROM TAX.V_GST_EXCEPTIONS,
             LATERAL FLATTEN(input => SPLIT(EXCEPTION_CODES, ', '))
        ORDER BY 1
    """).to_pandas()["EXCEPTION_CODE"].tolist()
    return entities, codes


@st.cache_data(ttl=600)
def load_exceptions(severity, entity, code, row_limit):
    clauses = []
    if severity != "All":
        clauses.append(f"MAX_SEVERITY = '{severity}'")
    if entity != "All":
        clauses.append(f"ENTITY_NAME = '{entity}'")
    if code != "All":
        clauses.append(f"EXCEPTION_CODES LIKE '%{code}%'")
    where = ("WHERE " + " AND ".join(clauses)) if clauses else ""
    return session.sql(f"""
        SELECT GL_LINE_ID, ENTITY_NAME, TAX_PERIOD, SOURCE_SYSTEM,
               SUPPLIER_NAME, SUPPLY_CATEGORY, TAX_TREATMENT, APPLIED_RULE_ID,
               NET_AMOUNT, BOOKED_TAX_AMOUNT, EXPECTED_TAX_AMOUNT, TAX_VARIANCE,
               EXCEPTION_CODES, MAX_SEVERITY, RISK_SCORE
        FROM TAX.V_GST_EXCEPTIONS
        {where}
        ORDER BY RISK_SCORE DESC
        LIMIT {row_limit}
    """).to_pandas()


@st.cache_data(ttl=600)
def load_exception_summary(severity, entity, code):
    clauses = []
    if severity != "All":
        clauses.append(f"MAX_SEVERITY = '{severity}'")
    if entity != "All":
        clauses.append(f"ENTITY_NAME = '{entity}'")
    if code != "All":
        clauses.append(f"EXCEPTION_CODES LIKE '%{code}%'")
    where = ("WHERE " + " AND ".join(clauses)) if clauses else ""
    return session.sql(f"""
        SELECT COUNT(*)                                  AS FLAGGED_LINES,
               ROUND(SUM(ABS(COALESCE(TAX_VARIANCE,0))),2) AS ABS_TAX_VARIANCE
        FROM TAX.V_GST_EXCEPTIONS
        {where}
    """).to_pandas()


@st.cache_data(ttl=600)
def load_heatmap():
    return session.sql("""
        SELECT ENTITY_NAME,
               TO_VARCHAR(TAX_PERIOD, 'YYYY-MM') AS TAX_PERIOD,
               COUNT(*)                          AS EXCEPTION_LINES
        FROM TAX.V_GST_EXCEPTIONS
        WHERE TAX_PERIOD IS NOT NULL
        GROUP BY 1, 2
        ORDER BY 1, 2
    """).to_pandas()


# ---------------------------------------------------------------------
# Headline tiles
# ---------------------------------------------------------------------
st.title("GST Tax Control Tower")
st.caption("Computed GST position, lodged BAS reconciliation and risk-ranked exception register.")

h = load_headline().iloc[0]
e = load_exception_headline().iloc[0]

c1, c2, c3, c4, c5 = st.columns(5)
c1.metric("Entity-periods", f"{int(h['ENTITY_PERIODS']):,}")
c2.metric("Matched", f"{int(h['MATCHED']):,}")
c3.metric("Investigate", f"{int(h['INVESTIGATE']):,}")
c4.metric("Flagged lines", f"{int(e['FLAGGED_LINES']):,}")
c5.metric("High severity", f"{int(e['HIGH_SEVERITY']):,}")

st.divider()

tab_recon, tab_exceptions, tab_heatmap = st.tabs(
    ["BAS reconciliation", "Exception register", "Entity heatmap"]
)


# ---------------------------------------------------------------------
# Tab 1 - BAS reconciliation
# ---------------------------------------------------------------------
with tab_recon:
    st.subheader("Computed GST against lodged BAS")

    status = st.selectbox(
        "Recon status",
        ["All", "MATCHED", "REVIEW", "INVESTIGATE"],
        key="recon_status",
    )

    st.markdown("**Total absolute variance by entity**")
    st.bar_chart(
        load_variance_by_entity().set_index("ENTITY_NAME"),
        y="TOTAL_ABS_VARIANCE",
        height=320,
    )

    recon = load_recon(status)
    st.markdown(f"**Entity-periods, worst variance first** ({len(recon):,} shown)")
    st.dataframe(
        recon.style.format({
            "COMPUTED_NET_GST": "{:,.2f}",
            "LODGED_NET_GST": "{:,.2f}",
            "VARIANCE_NET": "{:,.2f}",
            "VARIANCE_PCT": "{:,.2f}",
        }),
        use_container_width=True,
        height=420,
    )


# ---------------------------------------------------------------------
# Tab 2 - Exception register
# ---------------------------------------------------------------------
with tab_exceptions:
    st.subheader("Risk-ranked exception register")

    entities, codes = load_filter_values()

    f1, f2, f3, f4 = st.columns(4)
    severity = f1.selectbox("Severity", ["All", "HIGH", "MEDIUM", "LOW"], key="ex_sev")
    entity = f2.selectbox("Entity", ["All"] + entities, key="ex_ent")
    code = f3.selectbox("Exception code", ["All"] + codes, key="ex_code")
    row_limit = f4.selectbox("Rows to return", [100, 250, 500, 1000], key="ex_rows")

    summary = load_exception_summary(severity, entity, code).iloc[0]
    s1, s2 = st.columns(2)
    s1.metric("Flagged lines matching filters", f"{int(summary['FLAGGED_LINES']):,}")
    s2.metric("Absolute tax variance", f"{float(summary['ABS_TAX_VARIANCE'] or 0):,.2f}")

    st.caption(
        "The register holds 253,394 flagged lines. It is always read "
        "ORDER BY RISK_SCORE DESC with a row limit - the ranking is the point, "
        "not the volume."
    )

    st.dataframe(
        load_exceptions(severity, entity, code, row_limit),
        use_container_width=True,
        height=460,
    )


# ---------------------------------------------------------------------
# Tab 3 - Entity heatmap
# ---------------------------------------------------------------------
with tab_heatmap:
    st.subheader("Exception count by entity and tax period")

    hm = load_heatmap()
    pivot = hm.pivot(index="ENTITY_NAME", columns="TAX_PERIOD", values="EXCEPTION_LINES")
    pivot = pivot.fillna(0).astype(int)

    st.dataframe(
        pivot.style.background_gradient(axis=None, cmap="Reds"),
        use_container_width=True,
        height=520,
    )

    st.caption(
        "Dark cells are entity-periods carrying the most flagged lines. "
        "Read down a column for a bad month across the group, and across "
        "a row for an entity with a persistent problem."
    )

> 💡 **Why every query aggregates or limits**
>
> The two views behind this app sit on 2,400,000 ledger lines and 253,394 flagged exceptions. Pull either of those into the browser row-level and the app stalls — and the user learns nothing, because nobody reads a quarter of a million rows.

> ✅ **What you did not have to do**
>
> There is no connection string, no credential, no driver, no `pip install`, no container, no web server and no deployment. `get_active_session()` returns the session you are already authenticated in, so the app inherits your role and therefore your data access — row access policies and masking policies continue to apply.

### Block 2 — only if the app cannot see the objects

If the app loads but every tab reports that an object does not exist, the role running the app is not the role that built the objects. Run the grants below as `ACCOUNTADMIN`, substituting the role your app runs under, then reload the app.

In [ ]:
-- Substitute the role the Streamlit app runs under.
SET app_role = 'SYSADMIN';

GRANT USAGE  ON DATABASE GSI_TAX_WORKSHOP           TO ROLE IDENTIFIER($app_role);
GRANT USAGE  ON SCHEMA   GSI_TAX_WORKSHOP.TAX       TO ROLE IDENTIFIER($app_role);
GRANT USAGE  ON SCHEMA   GSI_TAX_WORKSHOP.CURATED   TO ROLE IDENTIFIER($app_role);
GRANT USAGE  ON SCHEMA   GSI_TAX_WORKSHOP.RAW_DATA  TO ROLE IDENTIFIER($app_role);

GRANT SELECT ON ALL VIEWS           IN SCHEMA GSI_TAX_WORKSHOP.TAX      TO ROLE IDENTIFIER($app_role);
GRANT SELECT ON ALL TABLES          IN SCHEMA GSI_TAX_WORKSHOP.TAX      TO ROLE IDENTIFIER($app_role);
GRANT SELECT ON ALL DYNAMIC TABLES  IN SCHEMA GSI_TAX_WORKSHOP.CURATED  TO ROLE IDENTIFIER($app_role);
GRANT SELECT ON ALL TABLES          IN SCHEMA GSI_TAX_WORKSHOP.RAW_DATA TO ROLE IDENTIFIER($app_role);

GRANT USAGE  ON WAREHOUSE GSI_TAX_WH TO ROLE IDENTIFIER($app_role);

> ✅ **Where you are**
>
> The same two views that produced a results grid in Step 5 are now a filterable application with headline tiles, a variance chart, a risk-ranked register and a heatmap — built in one file, running next to the data, and shareable with a URL. This is the visual payoff of the whole lab.

---

## Step 7 — The tax team sends you a rate change

*~10 min · OPTIONAL* · [narrative & expected results](gsi_tax_workshop_lab1_foundation.html#step-7)

This is how tax logic actually arrives. Not as a change request, not as a specification, not through a project — as a spreadsheet, by email, from the person who owns the rule. Following the imported-services review in Step 4, the tax team wants the same reverse-charge correction extended to another supply category from the start of next quarter, and they have asked you what it does to the group position *before* it happens.

- Load a tax team spreadsheet into a staging table through the Snowsight interface.
- Clone the entire environment in one statement, at no storage cost.
- Apply the new rules to the clone only, closing off the row they supersede.
- Compare the group GST position across both databases in a single query.
- Throw the scenario away.

### Upload the spreadsheet

No code for this part. The file is `assets/gsi_tax_rule_update.csv` from the workshop bundle — three determination rows, two of them with a `VALID_FROM` in the next quarter.

1. In the left navigation choose **Data**, then **Databases**.
2. Expand `GSI_TAX_WORKSHOP` and select the `TAX` schema.
3. Click **Create**, then **Table**, then **From File**.
4. Drop `gsi_tax_rule_update.csv` into the upload panel.
5. Name the table `STG_RULE_UPDATE`.
6. Snowsight infers the column names and types from the header row — check that `VALID_FROM` and `VALID_TO` came through as `DATE` and that the two boolean columns came through as `BOOLEAN`, then click **Load**.

The tax team's spreadsheet is now a governed table in the `TAX` schema, with inferred types, visible to anyone with access and versioned by the platform. Nobody wrote a loader.

> 💡 **What just changed about that spreadsheet**
>
> It was an attachment in somebody's inbox: no history, no access control, no lineage, and a different copy on four laptops. It is now a table.

### Block 1 — clone the entire environment

You are about to change tax rules. Not in production.

In [ ]:
CREATE DATABASE GSI_TAX_SCENARIO CLONE GSI_TAX_WORKSHOP;

In [ ]:
-- Prove the clone is complete: the ledger, the rules and the
-- reconciliation are all there.
SELECT (SELECT COUNT(*) FROM GSI_TAX_SCENARIO.CURATED.GL_HARMONISED)          AS LEDGER_LINES,
       (SELECT COUNT(*) FROM GSI_TAX_SCENARIO.TAX.TAX_RULE_DETERMINATION)     AS RULES,
       (SELECT COUNT(*) FROM GSI_TAX_SCENARIO.TAX.V_GST_VS_BAS)               AS ENTITY_PERIODS;

Every schema, every table, every view and all 2,400,000 ledger lines, in one statement. It completes in seconds and consumes no storage, because the clone shares the underlying micro-partitions with the original until one of them diverges. From this point the two databases are independent: anything you do in `GSI_TAX_SCENARIO` is invisible to `GSI_TAX_WORKSHOP`.

### Block 2 — apply the new rules to the clone only

Note the database qualification on every statement. This is the whole safety mechanism, and it is worth reading twice before you run it.

In [ ]:
-- 0. Fallback, so this notebook runs end to end even if you skipped the
--    upload. Uploading the CSV through the UI is the point of the exercise -
--    that is how a rule change actually arrives - but the staging table has
--    to exist either way. This creates it only if it is missing.
CREATE TABLE IF NOT EXISTS GSI_TAX_WORKSHOP.TAX.STG_RULE_UPDATE (
    RULE_ID                 NUMBER,
    PRIORITY                NUMBER,
    JURISDICTION            VARCHAR,
    SUPPLY_CATEGORY         VARCHAR,
    SUPPLIER_GST_REGISTERED BOOLEAN,
    IS_CROSS_BORDER         BOOLEAN,
    TAX_TREATMENT           VARCHAR,
    RECOVERABLE             BOOLEAN,
    VALID_FROM              DATE,
    VALID_TO                DATE,
    RULE_DESCRIPTION        VARCHAR
);

INSERT INTO GSI_TAX_WORKSHOP.TAX.STG_RULE_UPDATE
SELECT * FROM VALUES
    (17,20,'AU','STANDARD_GOODS'   ,FALSE,TRUE,'REVERSE_CHARGE',TRUE,'2026-10-01'::DATE,'9999-12-31'::DATE,'Imported standard goods from an unregistered offshore supplier - reverse charge applies from 1 Oct 2026'),
    (18,20,'AU','CAPITAL_EQUIPMENT',FALSE,TRUE,'REVERSE_CHARGE',TRUE,'2026-10-01'::DATE,'9999-12-31'::DATE,'Imported capital equipment from an unregistered offshore supplier - reverse charge applies from 1 Oct 2026'),
    (19,40,'AU','STANDARD_GOODS'   ,FALSE,TRUE,'TAXABLE'       ,TRUE,'2000-07-01'::DATE,'2026-09-30'::DATE,'Superseded - imported standard goods treated as ordinary taxable supply until 30 Sep 2026')
WHERE NOT EXISTS (SELECT 1 FROM GSI_TAX_WORKSHOP.TAX.STG_RULE_UPDATE);

In [ ]:
-- 1. Insert the tax team's new determination rows into the CLONE.
INSERT INTO GSI_TAX_SCENARIO.TAX.TAX_RULE_DETERMINATION
    (RULE_ID, PRIORITY, JURISDICTION, SUPPLY_CATEGORY, SUPPLIER_GST_REGISTERED,
     IS_CROSS_BORDER, TAX_TREATMENT, RECOVERABLE, VALID_FROM, VALID_TO, RULE_DESCRIPTION)
SELECT RULE_ID, PRIORITY, JURISDICTION, SUPPLY_CATEGORY, SUPPLIER_GST_REGISTERED,
       IS_CROSS_BORDER, TAX_TREATMENT, RECOVERABLE, VALID_FROM, VALID_TO, RULE_DESCRIPTION
FROM GSI_TAX_WORKSHOP.TAX.STG_RULE_UPDATE;

-- 2. Close off the row the new position supersedes. The existing rule is
--    not deleted and not edited - its VALID_TO is set to the day before
--    the new rule starts, so the historical answer is preserved.
UPDATE GSI_TAX_SCENARIO.TAX.TAX_RULE_DETERMINATION
   SET VALID_TO = DATEADD('day', -1, DATE '2026-10-01')
 WHERE RULE_ID = 11
   AND VALID_TO = DATE '9999-12-31';

In [ ]:
-- 3. Read back the rules that now govern imported goods.
SELECT RULE_ID, PRIORITY, SUPPLY_CATEGORY, SUPPLIER_GST_REGISTERED,
       IS_CROSS_BORDER, TAX_TREATMENT, RECOVERABLE, VALID_FROM, VALID_TO
FROM GSI_TAX_SCENARIO.TAX.TAX_RULE_DETERMINATION
WHERE SUPPLY_CATEGORY IN ('STANDARD_GOODS', 'CAPITAL_EQUIPMENT')
ORDER BY SUPPLY_CATEGORY, PRIORITY, VALID_FROM;

Read what statement 2 does not do. It does not delete rule 11 and it does not change its treatment. It sets its `VALID_TO`, so a line dated before 1 October 2026 still resolves to the old position and a line dated after it resolves to the new one. That is the difference between a rules table and a configuration file: the historical answer remains reproducible, which is exactly what you need when the auditor asks why a transaction from last year was treated the way it was.

> ⚠️ **No rebuild, no reload, no redeploy**
>
> `TAX.V_GST_DETERMINED` is a view, so the moment those rows land the determined treatment for every affected line has already changed in the clone. There is nothing to refresh, nothing to recompute and nothing to deploy — the next query simply returns a different and correct answer.

### Block 3 — the two positions, side by side

In [ ]:
WITH production AS (
    SELECT ENTITY_NAME,
           SUM(COMPUTED_NET_GST) AS NET_GST
    FROM GSI_TAX_WORKSHOP.TAX.V_GST_VS_BAS
    GROUP BY ENTITY_NAME
),
scenario AS (
    SELECT ENTITY_NAME,
           SUM(COMPUTED_NET_GST) AS NET_GST
    FROM GSI_TAX_SCENARIO.TAX.V_GST_VS_BAS
    GROUP BY ENTITY_NAME
)
SELECT COALESCE(p.ENTITY_NAME, s.ENTITY_NAME)                     AS ENTITY_NAME,
       ROUND(COALESCE(p.NET_GST, 0), 2)                           AS NET_GST_TODAY,
       ROUND(COALESCE(s.NET_GST, 0), 2)                           AS NET_GST_UNDER_NEW_RULES,
       ROUND(COALESCE(s.NET_GST, 0) - COALESCE(p.NET_GST, 0), 2)   AS IMPACT,
       ROUND((COALESCE(s.NET_GST, 0) - COALESCE(p.NET_GST, 0))
             / NULLIF(ABS(COALESCE(p.NET_GST, 0)), 0) * 100, 2)    AS IMPACT_PCT
FROM production p
FULL OUTER JOIN scenario s ON s.ENTITY_NAME = p.ENTITY_NAME
ORDER BY ABS(COALESCE(s.NET_GST, 0) - COALESCE(p.NET_GST, 0)) DESC;

One query, two databases, fourteen entities, and the answer the tax team asked for. Because the new rules carry a future `VALID_FROM`, the impact on *historical* periods is nil by construction — which is itself the finding, and the point worth making out loud: no lodged period is restated. The forward effect appears the moment those periods close, and the tax team can see the number now rather than discovering it in a quarterly true-up.

### Block 4 — throw the scenario away

In [ ]:
DROP DATABASE GSI_TAX_SCENARIO;

Gone. The production database was never touched, and if you decide tomorrow that you needed the scenario after all, `UNDROP DATABASE GSI_TAX_SCENARIO;` brings it back within the retention window.

> ✅ **What the exercise actually cost**
>
> The tax team's spreadsheet became governed, effective-dated reference data. Modelling its impact across a 2,400,000-line ledger cost one clone and four minutes — no environment request, no data copy, no storage, production untouched and no lodged period restated.

> 💡 **Why this is the pattern, not a trick**
>
> Every rate change, every treaty update and every legislative amendment is this exercise. A rule arrives from the person who owns it, somebody has to say what it does to the numbers, and the answer has to be produced without disturbing the position already lodged.

---

## Step 8 — Share the snapshot with the auditor

*~6 min · OPTIONAL* · [narrative & expected results](gsi_tax_workshop_lab1_foundation.html#step-8)

The audit snapshot from Step 5 exists and it is immutable. The auditor now needs to read it. The usual answer is an extract — a CSV, a secure file transfer, a copy of the ledger that leaves your control the moment it is sent and is stale before it arrives. There is a better one.

- Create a share and grant it access to the audit snapshot and nothing else.
- Verify exactly what the consumer can see before anybody is added to it.
- Understand why no data is copied and why access remains revocable.

### Block 1 — create the share and grant it the snapshot only

Read the grant chain carefully. Usage on the database, usage on the `AUDIT` schema, select on one table. There is no grant on `RAW_DATA`, none on `CURATED` and none on `TAX` — so the ledger, the supplier master, the rules and the exception register are all outside the share. The auditor sees the snapshot and nothing else.

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE SHARE GST_AUDIT_SNAPSHOT_SHARE
  COMMENT = 'As-lodged GL snapshot for external audit. Read only.';

-- Usage on the container objects. This grants the ability to see the
-- database and the schema exist - it does not grant access to any data.
GRANT USAGE ON DATABASE GSI_TAX_WORKSHOP        TO SHARE GST_AUDIT_SNAPSHOT_SHARE;
GRANT USAGE ON SCHEMA   GSI_TAX_WORKSHOP.AUDIT  TO SHARE GST_AUDIT_SNAPSHOT_SHARE;

-- The only object in the share: the immutable snapshot from Step 5.
-- Nothing in RAW_DATA, CURATED or TAX is granted.
GRANT SELECT ON TABLE GSI_TAX_WORKSHOP.AUDIT.GL_AS_LODGED_20260803
  TO SHARE GST_AUDIT_SNAPSHOT_SHARE;

### Block 2 — verify what the consumer can actually see

In [ ]:
SHOW GRANTS TO SHARE GST_AUDIT_SNAPSHOT_SHARE;

Three rows: `USAGE` on the database, `USAGE` on the schema, `SELECT` on one table. That output *is* the scope of the share — there is no second place to look and no inherited access to reason about. Run it before you add anybody, and run it again whenever somebody asks what the auditor can see.

> 💡 **No data is copied**
>
> The consumer queries your micro-partitions directly, from their own account, with their own warehouse paying for their own compute. Nothing is extracted, nothing is transferred and there is no second copy to secure, reconcile or later destroy.

The snapshot in this share is a clone taken at a fixed point in time, so its contents will not change — that is deliberate for an audit. The properties of the share itself are what matter: read-only, live against the granted object, revocable in one statement, and precisely scoped to objects you named explicitly.

### Adding the consumer account

One statement completes the share, and it needs something this workshop cannot supply:

> ⚠️ **ALTER SHARE ... ADD ACCOUNTS needs a real account identifier**
>
> The statement is `ALTER SHARE GST_AUDIT_SNAPSHOT_SHARE ADD ACCOUNTS = <organisation>.<account>;` — for example `ADD ACCOUNTS = EXAMPLE_ORG.AUDITOR_ACCOUNT_PLACEHOLDER`, which is an obvious placeholder and will fail. Do not run it with a guessed value.

> ✅ **What you would have done instead**
>
> Exported 2,400,000 rows to a file, agreed a transfer mechanism with the auditor, obtained approval to send client ledger data outside the organisation, and then had no way to withdraw it, no way to tell whether it had been copied again, and no answer when the auditor asked three weeks later whether their extract was still current. A share replaces …